# 00. Project Overview and Research Hypothesis
## Retrieval-Enhanced HATNet for Remote Sensing Image Captioning

**Primary Models Evaluated:**
- **Model A (`baseline`)**: Original HATNet (`GeoRSCLIP ViT-H/14` multi-layer `[10, 21, 31]` features + 3-level Hierarchical Encoder + Pre-encoded Text Encoder (PTE) + Channel/Spatial Hierarchical Attention Decoder). `USE_RETRIEVAL = False`.
- **Model B (`retrieval_only`)**: Original HATNet + Training-Only Caption Retrieval Conditioning (similarity-weighted token embedding aggregation fused via gated linear projection, without the ViPCap query-token mapper).
- **Model C (`full_vipcap`)**: Original HATNet + Training-Only Caption Retrieval + ViPCap-Inspired Visual Prompt Mapper (`ViPCapPromptMapper` with learnable prompt tokens cross-attending to retrieved caption tokens) + Gated Spatial-Semantic Prompt Fusion (`VisualPromptFusion`).

**Core Research Hypothesis:**
Remote sensing imagery in RSICD contains dense, visually ambiguous scenes (e.g., distinguishing *baseballfield* from *mediumresidential* or counting fine-grained structures). While GeoRSCLIP ViT-H/14 patch tokens (`[256, 3840]`) capture local visual appearance, a training-only retrieval memory provides reusable domain-specific semantic priors from visually similar training scenes. Transforming retrieved training captions into a visual prompt and fusing it before HATNet's hierarchical spatial/channel attention (`DecoderLayer2`) may help the decoder resolve semantic ambiguity—a hypothesis we test rigorously without assuming positive gains a priori.

# 01. Existing Implementation Audit

### Verified Architecture & Bugs Fixed from `HATNet_RSICD_Baseline(1).ipynb` and `hanhan68/HATNet`:
1. **Existing Tensor Dimensions**:
   - GeoRSCLIP `ViT-H-14` (`RS5M_ViT-H-14.pt`) extracts 256 patch tokens (`16x16` grid, dropping CLS index 0) from ResBlocks `[10, 21, 31]`, each `1280`-d, concatenated to `(B, 256, 3840)`.
   - `MultiLevelAttEmbed` projects each `1280`-d block into `d_model = 512`, yielding `(B, 256, 1536)`.
   - `EncoderDecoder.encode()` slices `0:512`, `512:1024`, and `1024:1536`, refines them through the hierarchical `Encoder`, and outputs `memory` of shape `(B, 256, 1536)`.
   - `Decoder1` (PTE) encodes target word embeddings `(B, L, 512)` into pre-encoded text representations `m` `(B, L, 512)`.
   - `Decoder2` (`DecoderLayer2`) computes $V_m = \text{FFN}_{\text{mul}}(\text{memory}) \cdot \eta + \text{memory}_{[:, :, 1024:1536]}$, reshapes $V_m$ to `(B, 512, 16, 16)`, applies `ChannelAttention(512, 512)` and `SpatialAttention()`, and gates their cross-attention outputs via `fc_alpha1` and `fc_alpha2`.
2. **Existing Bugs & Required Corrections Implemented Here**:
   - **Bug 1 (Upstream `DecoderLayer2.forward`)**: `ChannelAttention`, `SpatialAttention`, and `yita` were instantiated inside `forward()` with `.to("cuda")`, preventing optimizer registration. *Fixed*: Registered in `DecoderLayer2.__init__`.
   - **Bug 2 (Broken `att_embed` Regex in `FINAL(2)_2.ipynb`)**: Duplicated closing regex caused `AssertionError: matched 0`. *Fixed*: Clean AST/regex replacement with count verification.
   - **Bug 3 (Incomplete Cache Validation)**: Previous notebook skipped metadata verification. *Fixed*: Full 12-field schema validation + 8 automated unit tests.
   - **Bug 4 (Inconsistent Evaluation Fallback)**: Previous notebook silently swapped tokenizers and dropped METEOR when Java/SPICE failed. *Fixed*: Single deterministic `ConsistentCaptionEvaluator` used identically across validation and test sets, isolating optional SPICE without altering BLEU/METEOR/ROUGE-L/CIDEr.

# 02. Research Architecture and Design Decisions

### Mathematical Formulation of Retrieval & Prompt Fusion
1. **Training-Only Memory Embedding ($e_i \in \mathbb{R}^{1280}$)**:
   For each training image $i \in \mathcal{D}_{\text{train}}$ ($\vert{}\mathcal{D}_{\text{train}}\vert{} = 2400$), we mean-pool the high-level GeoRSCLIP block-31 patch tokens $F^{(31)}_i \in \mathbb{R}^{256 \times 1280}$ and $\ell_2$-normalize:
   $$e_i = \frac{\frac{1}{256}\sum_{p=1}^{256} F^{(31)}_{i, p}}{\left\|\frac{1}{256}\sum_{p=1}^{256} F^{(31)}_{i, p}\right\|_2}$$
2. **Leakage-Free Top-$K$ Retrieval**:
   Given query image $q$ with normalized embedding $e_q$, cosine similarity $s_{q, i} = e_q^\top e_i$ is computed strictly over $i \in \mathcal{D}_{\text{train}} \setminus \{q\}$ (explicitly masking $s_{q, q} = -\infty$ during training). Top-$K$ neighbors $(i_1, \dots, i_K)$ and temperature-scaled softmax weights $w_k = \frac{\exp(s_{q, i_k} / \tau)}{\sum_{j=1}^K \exp(s_{q, i_j} / \tau)}$ are returned.
3. **Retrieved-Caption Encoding & ViPCap-Inspired Prompt Mapping**:
   Each neighbor $i_k$ has up to $M=5$ training captions tokenized with the training-only vocabulary. Token embeddings are masked-mean-pooled per caption, averaged per neighbor, and weighted by $w_k$ to form both a sequence of retrieved token states $T_q \in \mathbb{R}^{B \times (K \cdot L_c) \times d}$ and a weighted summary vector $r_q \in \mathbb{R}^{B \times d}$.
   - In **Model B (`retrieval_only`)**, $r_q$ is projected and broadcast-gated onto the high-level visual tokens.
   - In **Model C (`full_vipcap`)**, $N_p = 16$ learnable visual-prompt queries $P_0 \in \mathbb{R}^{B \times N_p \times d}$ cross-attend to $T_q$ (weighted by similarity scores) through `ViPCapPromptMapper` to produce visual prompt tokens $P_q \in \mathbb{R}^{B \times N_p \times d}$.
4. **Insertion Location & Gated Fusion (`VisualPromptFusion`)**:
   Inserted inside `EncoderDecoder.encode()` on the encoded multi-level memory $M \in \mathbb{R}^{B \times 256 \times 1536}$ immediately before `DecoderLayer2`'s hierarchical spatial/channel attention. Specifically, the semantic slice $M^{(3)} = M_{[:, :, 1024:1536]} \in \mathbb{R}^{B \times 256 \times 512}$ cross-attends to $P_q$ and updates via a learnable gate initialized near zero:
   $$M^{(3)}_{\text{enhanced}} = M^{(3)} + \tanh(\alpha) \cdot \sigma(W_g [M^{(3)}; \text{CrossAttn}(M^{(3)}, P_q, P_q)]) \odot \text{CrossAttn}(M^{(3)}, P_q, P_q)$$
   This preserves the exact `[B, 256, 1536]` layout expected by `DecoderLayer2` and guarantees that when `USE_RETRIEVAL = False`, the computation is bit-for-bit identical to the original HATNet baseline.

In [1]:
# 03. Configuration and Reproducibility
import os
import sys
import json
import time
import math
import copy
import shutil
import random
import hashlib
import datetime
import traceback
import subprocess
import py_compile
from pathlib import Path
from types import SimpleNamespace
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
# Suppress Hugging Face symlink warnings on Windows
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

def set_all_seeds(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

class ExperimentConfig:
    # --- Reproducibility & Environment ---
    SEED = 42
    IS_KAGGLE = os.path.exists("/kaggle")
    
    # --- Primary Output Directory (Desktop\vansh for local PC, /kaggle/working for Kaggle) ---
    LOCAL_OUTPUT_DIR = os.path.join(os.path.expanduser("~"), "Desktop", "vansh")
    WORKSPACE_DIR = "/kaggle/working" if IS_KAGGLE else LOCAL_OUTPUT_DIR
    REPO_DIR = os.path.join(WORKSPACE_DIR, "HATNet")
    
    # --- Output Directories & Artifact Paths ---
    OUT_DATA_DIR = os.path.join(WORKSPACE_DIR, "data")
    FEATURES_DIR = os.path.join(WORKSPACE_DIR, "features")
    CHECKPOINT_DIR = os.path.join(WORKSPACE_DIR, "checkpoints")
    RESULTS_DIR = os.path.join(WORKSPACE_DIR, "results")
    
    MANIFEST_FILE = os.path.join(WORKSPACE_DIR, "split_manifest.json")
    CONFIG_JSON = os.path.join(WORKSPACE_DIR, "configuration.json")
    FEATURE_CACHE_META_JSON = os.path.join(WORKSPACE_DIR, "feature_cache_metadata.json")
    RETRIEVAL_META_JSON = os.path.join(WORKSPACE_DIR, "retrieval_memory_metadata.json")
    TRAINING_HISTORY_JSON = os.path.join(WORKSPACE_DIR, "training_history.json")
    VAL_PREDS_JSON = os.path.join(WORKSPACE_DIR, "validation_predictions.json")
    TEST_PREDS_JSON = os.path.join(WORKSPACE_DIR, "test_predictions.json")
    TEST_METRICS_JSON = os.path.join(WORKSPACE_DIR, "test_metrics.json")
    RUN_SUMMARY_TXT = os.path.join(WORKSPACE_DIR, "run_summary.txt")
    DATASET_3000_JSON = os.path.join(OUT_DATA_DIR, "dataset_3000.json")
    
    # --- Dataset Split Specifications ---
    TOTAL_IMAGES = 3000               # 2,400 Train / 300 Val / 300 Test
    TRAIN_COUNT = 2400
    VAL_COUNT = 300
    TEST_COUNT = 300
    MAX_CAPTION_LEN = 20
    MIN_WORD_FREQ = 2
    SEQ_PER_IMG = 5
    
    # --- GeoRSCLIP Feature Extraction Specifications ---
    MODEL_NAME = "ViT-H-14"
    EXPECTED_CKPT_SHA256 = "62a3e79df886875976901b4a9b8d6a1c42b457879a5ce625c8eba9460cbc3a11"
    ALLOW_CKPT_HASH_OVERRIDE = True   # Resilient against minor packaging variations
    EXTRACTION_LAYERS = [10, 21, 31]
    NUM_PATCHES = 256
    ATT_FEAT_SIZE = 3840
    CONCAT_ORDER = "channels_10_21_31"
    CACHE_SCHEMA_VERSION = "2.0.0"
    EXTRACT_BATCH_SIZE = 16
    
    # --- HATNet Architecture Hyperparameters ---
    CAPTION_MODEL = "transformer"
    D_MODEL = 512
    D_FF = 2048
    NUM_HEADS = 8
    N_ENC = 3
    N_DEC = 3
    DROPOUT = 0.1
    DROP_PROB_LM = 0.1
    
    # --- Retrieval & ViPCap Visual Prompt Hyperparameters ---
    USE_RETRIEVAL = True              # Set False for pure Model A (Original HATNet Baseline)
    AB_MODE = "full_vipcap"           # Options: "baseline" (Model A), "retrieval_only" (Model B), "full_vipcap" (Model C)
    RETRIEVAL_TOP_K = 4
    RETRIEVAL_SIMILARITY = "cosine"
    RETRIEVAL_TEMPERATURE = 0.07
    RETRIEVAL_SCORE_THRESHOLD = -1.0
    RETRIEVAL_MEMORY_MODE = "block31_mean_norm"
    NUM_PROMPT_TOKENS = 16
    PROMPT_DROPOUT = 0.1
    
    # --- Training & Evaluation Hyperparameters ---
    BATCH_SIZE = 16
    LEARNING_RATE = 3e-4
    WARMUP_STEPS = 450
    MAX_EPOCHS = 30
    GRAD_CLIP_NORM = 0.1
    BEAM_SIZE = 3
    VAL_IMAGES_USE = 300
    RUN_FULL_TRAINING = True          # Configurable switch
    RUN_OPTIONAL_ABLATIONS = False    # Prevents unauthorized expensive multi-run ablations

    @classmethod
    def to_dict(cls):
        return {
            k: v for k, v in cls.__dict__.items()
            if not k.startswith("_") and isinstance(v, (int, float, str, bool, list, dict, tuple))
        }

set_all_seeds(ExperimentConfig.SEED)
for d in [ExperimentConfig.WORKSPACE_DIR, ExperimentConfig.OUT_DATA_DIR,
          ExperimentConfig.FEATURES_DIR, ExperimentConfig.CHECKPOINT_DIR,
          ExperimentConfig.RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

with open(ExperimentConfig.CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(ExperimentConfig.to_dict(), f, indent=2)
print(f"Workspace Directory : {ExperimentConfig.WORKSPACE_DIR}")
print(f"Configuration saved to {ExperimentConfig.CONFIG_JSON}")


Workspace Directory : C:\Users\LENOVO\Desktop\vansh
Configuration saved to C:\Users\LENOVO\Desktop\vansh\configuration.json


In [2]:
# 04. Dependency Installation
import sys
import subprocess
print("Installing and verifying required dependencies...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "open_clip_torch", "h5py", "yacs", "pycocotools", "pycocoevalcap",
    "lmdbdict", "tqdm", "datasets", "huggingface_hub", "nltk", "transformers"
], check=True)

import h5py
import open_clip
import yacs
import transformers
from tqdm import tqdm
from PIL import Image
from datasets import load_dataset, concatenate_datasets
from huggingface_hub import hf_hub_download
print("All required packages imported successfully.")


Installing and verifying required dependencies...
All required packages imported successfully.


In [3]:
# 05. GPU and Environment Verification
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version : {torch.__version__}")
print(f"Active Device   : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU Name        : {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU Total VRAM  : {vram_gb:.2f} GB")

total_b, used_b, free_b = shutil.disk_usage(ExperimentConfig.WORKSPACE_DIR)
raw_feat_bytes = ExperimentConfig.TOTAL_IMAGES * ExperimentConfig.NUM_PATCHES * ExperimentConfig.ATT_FEAT_SIZE * 4
raw_feat_gb = raw_feat_bytes / (1024 ** 3)
free_gb = free_b / (1024 ** 3)
print(f"Workspace Dir   : {ExperimentConfig.WORKSPACE_DIR}")
print(f"Disk Space Free : {free_gb:.2f} GB (Uncompressed 3,000-image feature size: {raw_feat_gb:.2f} GB)")
assert free_b > raw_feat_bytes * 0.45, f"Insufficient disk storage: {free_gb:.2f} GB free."


PyTorch Version : 2.11.0+cu128
Active Device   : cuda
GPU Name        : NVIDIA GeForce RTX 4050 Laptop GPU
GPU Total VRAM  : 6.00 GB
Workspace Dir   : C:\Users\LENOVO\Desktop\vansh
Disk Space Free : 250.75 GB (Uncompressed 3,000-image feature size: 10.99 GB)


In [4]:
# 06. Dataset Discovery & GeoRSCLIP Checkpoint Resolution
import re

def compute_file_sha256(filepath: str) -> str:
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(1024 * 1024):
            h.update(chunk)
    return h.hexdigest()

def normalize_caption_tokens(raw_text: str):
    cleaned = re.sub(r"[^a-z0-9\s]+", " ", str(raw_text).lower())
    return [t for t in cleaned.split() if t]

def discover_assets():
    # Targeted safe search roots (avoids scanning entire user profile or AppData/Anaconda trees)
    search_roots = [
        ExperimentConfig.WORKSPACE_DIR,
        os.path.join(os.path.expanduser("~"), "Desktop", "vansh"),
        os.path.join(os.path.expanduser("~"), "Downloads"),
    ]
    if ExperimentConfig.IS_KAGGLE:
        search_roots.extend(["/kaggle/input", "/kaggle/working"])
        
    existing_roots = list(dict.fromkeys([os.path.abspath(r) for r in search_roots if os.path.exists(r)]))
    cand_ckpts, cand_jsons, cand_img_dirs = [], [], set()
    
    ignore_dirs = {
        "appdata", "anaconda3", "miniconda3", "site-packages", ".git", ".gemini",
        "node_modules", ".cache", "$recycle.bin", ".ipynb_checkpoints"
    }
    
    for root in existing_roots:
        root_depth = root.rstrip(os.path.sep).count(os.path.sep)
        for dirpath, dirs, filenames in os.walk(root):
            # Prune directories: skip ignored system/package dirs and limit depth to 3
            dirs[:] = [d for d in dirs if d.lower() not in ignore_dirs and not d.startswith(".")]
            cur_depth = dirpath.rstrip(os.path.sep).count(os.path.sep) - root_depth
            if cur_depth > 3:
                del dirs[:]
                
            for f in filenames:
                if f == "RS5M_ViT-H-14.pt" or (f.endswith(".pt") and "ViT-H-14" in f):
                    cand_ckpts.append(os.path.abspath(os.path.join(dirpath, f)))
                if f.endswith(".json"):
                    low_f = f.lower()
                    if any(ex in low_f for ex in [
                        "manifest", "metrics", "pred", "dummy", "info.json", "configuration",
                        "eval_results", "metadata", "history", "dataset_3000", "rsicd_train",
                        "rsicd_val", "rsicd_test"
                    ]):
                        continue
                    if "rsicd" in low_f or "dataset" in low_f or "caption" in low_f:
                        cand_jsons.append(os.path.abspath(os.path.join(dirpath, f)))
            if sum(1 for f in filenames if f.lower().endswith((".jpg", ".jpeg", ".png"))) >= 20:
                cand_img_dirs.add(os.path.abspath(dirpath))
                
    cand_ckpts = list(dict.fromkeys(cand_ckpts))
    cand_jsons = list(dict.fromkeys(cand_jsons))
    
    # Load RSICD from Hugging Face Hub ('arampacha/rsicd')
    print("Loading RSICD dataset from Hugging Face ('arampacha/rsicd')...")
    hf_raw = load_dataset("arampacha/rsicd")
    hf_dataset = concatenate_datasets([hf_raw[s] for s in hf_raw.keys()]) if hasattr(hf_raw, "keys") else hf_raw
    hf_image_map = {os.path.basename(fn): idx for idx, fn in enumerate(hf_dataset["filename"])}
    print(f"Indexed {len(hf_image_map)} unique RSICD images from Hugging Face.")
    
    if not cand_jsons:
        exported_json = os.path.join(ExperimentConfig.OUT_DATA_DIR, "dataset_rsicd_karpathy.json")
        print(f"Exporting Karpathy-schema RSICD annotations from HF dataset to {exported_json}...")
        records = []
        for idx in range(len(hf_dataset)):
            row = hf_dataset[idx]
            fn = os.path.basename(row["filename"])
            sents = []
            for s_idx, cap in enumerate(row.get("captions", [])):
                toks = normalize_caption_tokens(cap)
                if toks:
                    sents.append({"raw": " ".join(toks), "tokens": toks, "imgid": idx, "sentid": s_idx})
            records.append({"filename": fn, "imgid": idx, "sentences": sents})
        with open(exported_json, "w", encoding="utf-8") as f:
            json.dump({"images": records, "dataset": "rsicd"}, f)
        cand_jsons.append(exported_json)
        
    resolved_json = cand_jsons[0]
    resolved_img_dir = list(cand_img_dirs)[0] if cand_img_dirs else None
    return resolved_json, cand_ckpts, hf_dataset, hf_image_map, resolved_img_dir

RESOLVED_JSON, CAND_CKPTS, HF_DATASET, HF_IMAGE_MAP, RESOLVED_IMG_DIR = discover_assets()


Loading RSICD dataset from Hugging Face ('arampacha/rsicd')...


Indexed 10921 unique RSICD images from Hugging Face.


In [5]:
# 07. Dataset Fingerprinting & Duplicate / Near-Duplicate Image Audit
def parse_and_fingerprint_rsicd(json_path: str, hf_image_map: dict, image_dir: str = None):
    annotation_sha256 = compute_file_sha256(json_path)
    with open(json_path, "r", encoding="utf-8") as f:
        raw_obj = json.load(f)
    raw_images = raw_obj["images"] if isinstance(raw_obj, dict) and "images" in raw_obj else raw_obj
    
    seen_ids, seen_fnames = set(), set()
    usable_records = []
    
    for entry in raw_images:
        fname = entry.get("filename")
        img_id = entry.get("imgid", entry.get("id"))
        if fname is None or img_id is None:
            continue
        base_fn = os.path.basename(str(fname))
        if int(img_id) in seen_ids or base_fn in seen_fnames:
            continue
        if base_fn not in hf_image_map and not (image_dir and os.path.exists(os.path.join(image_dir, fname))):
            continue
            
        sents = []
        for s in entry.get("sentences", []):
            raw_c = s.get("raw", s.get("caption", "")) if isinstance(s, dict) else str(s)
            toks = normalize_caption_tokens(raw_c)
            if toks:
                sents.append({
                    "raw": " ".join(toks),
                    "tokens": toks,
                    "imgid": int(img_id),
                    "sentid": len(sents)
                })
        if not sents:
            continue
            
        seen_ids.add(int(img_id))
        seen_fnames.add(base_fn)
        usable_records.append({
            "imgid": int(img_id),
            "filename": base_fn,
            "sentences": sents,
            "raw_captions": [c["raw"] for c in sents]
        })
        
    usable_records.sort(key=lambda r: r["filename"])
    dataset_fp_payload = "|".join(f"{r['imgid']}:{r['filename']}:{len(r['sentences'])}" for r in usable_records)
    dataset_fingerprint = hashlib.sha256(dataset_fp_payload.encode("utf-8")).hexdigest()
    
    print(f"Usable unique images audited : {len(usable_records)}")
    print(f"Annotation SHA-256           : {annotation_sha256}")
    print(f"Dataset Catalog Fingerprint  : {dataset_fingerprint}")
    assert len(usable_records) >= ExperimentConfig.TOTAL_IMAGES
    return usable_records, dataset_fingerprint, annotation_sha256

USABLE_RECORDS, DATASET_FINGERPRINT, ANNOTATION_FINGERPRINT = parse_and_fingerprint_rsicd(
    RESOLVED_JSON, HF_IMAGE_MAP, RESOLVED_IMG_DIR
)

Usable unique images audited : 10921
Annotation SHA-256           : 732aac3d0992a67eafb989cb02da79e038e2044849fbee4c9d05b834db4a9a7b
Dataset Catalog Fingerprint  : 1230c8994bc5910c25bf01b3ef9e878989115398349ab17b261447207bfca47e


In [6]:
# 08. Deterministic Dataset Splits (2,400 Train / 300 Val / 300 Test)
def build_or_verify_split_manifest(usable_records, manifest_path, dataset_fp, ann_fp):
    usable_by_id = {r["imgid"]: r for r in usable_records}
    if os.path.exists(manifest_path):
        try:
            with open(manifest_path, "r", encoding="utf-8") as f:
                m = json.load(f)
            t_ids, v_ids, te_ids = set(m["train_ids"]), set(m["val_ids"]), set(m["test_ids"])
            if (m.get("seed") == ExperimentConfig.SEED and
                m.get("dataset_fingerprint") == dataset_fp and
                len(t_ids) == ExperimentConfig.TRAIN_COUNT and
                len(v_ids) == ExperimentConfig.VAL_COUNT and
                len(te_ids) == ExperimentConfig.TEST_COUNT and
                t_ids.isdisjoint(v_ids) and t_ids.isdisjoint(te_ids) and v_ids.isdisjoint(te_ids) and
                all(i in usable_by_id for i in (t_ids | v_ids | te_ids))):
                print(f"[MANIFEST VERIFIED] Reusing valid split manifest at {manifest_path}")
                return m
        except Exception as e:
            print(f"Regenerating manifest due to validation mismatch: {e}")
            
    rng = random.Random(ExperimentConfig.SEED)
    indices = list(range(len(usable_records)))
    rng.shuffle(indices)
    selected = [usable_records[i] for i in indices[:ExperimentConfig.TOTAL_IMAGES]]
    
    train_recs = selected[:ExperimentConfig.TRAIN_COUNT]
    val_recs = selected[ExperimentConfig.TRAIN_COUNT : ExperimentConfig.TRAIN_COUNT + ExperimentConfig.VAL_COUNT]
    test_recs = selected[ExperimentConfig.TRAIN_COUNT + ExperimentConfig.VAL_COUNT :]
    
    split_fp_str = (
        f"{dataset_fp}:{ann_fp}:{ExperimentConfig.SEED}:" +
        ",".join(str(r["imgid"]) for r in selected)
    )
    split_fingerprint = hashlib.sha256(split_fp_str.encode("utf-8")).hexdigest()
    
    manifest = {
        "seed": ExperimentConfig.SEED,
        "dataset_fingerprint": dataset_fp,
        "annotation_fingerprint": ann_fp,
        "split_fingerprint": split_fingerprint,
        "total_images": ExperimentConfig.TOTAL_IMAGES,
        "train_count": len(train_recs),
        "val_count": len(val_recs),
        "test_count": len(test_recs),
        "train_ids": [r["imgid"] for r in train_recs],
        "val_ids": [r["imgid"] for r in val_recs],
        "test_ids": [r["imgid"] for r in test_recs],
        "train_filenames": [r["filename"] for r in train_recs],
        "val_filenames": [r["filename"] for r in val_recs],
        "test_filenames": [r["filename"] for r in test_recs],
        "train_captions": {str(r["imgid"]): r["raw_captions"] for r in train_recs},
        "val_captions": {str(r["imgid"]): r["raw_captions"] for r in val_recs},
        "test_captions": {str(r["imgid"]): r["raw_captions"] for r in test_recs}
    }
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
    print(f"Created and verified persistent split manifest: {manifest_path}")
    return manifest

SPLIT_MANIFEST = build_or_verify_split_manifest(
    USABLE_RECORDS, ExperimentConfig.MANIFEST_FILE, DATASET_FINGERPRINT, ANNOTATION_FINGERPRINT
)

TRAIN_IDS_SET = set(SPLIT_MANIFEST["train_ids"])
VAL_IDS_SET = set(SPLIT_MANIFEST["val_ids"])
TEST_IDS_SET = set(SPLIT_MANIFEST["test_ids"])
assert len(TRAIN_IDS_SET) == 2400 and len(VAL_IDS_SET) == 300 and len(TEST_IDS_SET) == 300
assert TRAIN_IDS_SET.isdisjoint(VAL_IDS_SET) and TRAIN_IDS_SET.isdisjoint(TEST_IDS_SET) and VAL_IDS_SET.isdisjoint(TEST_IDS_SET)

# Export canonical dataset_3000.json in exact split order
USABLE_MAP = {r["imgid"]: r for r in USABLE_RECORDS}
SELECTED_IMAGES = []
for split_name, id_list in [("train", SPLIT_MANIFEST["train_ids"]), ("val", SPLIT_MANIFEST["val_ids"]), ("test", SPLIT_MANIFEST["test_ids"])]:
    for iid in id_list:
        rec = copy.deepcopy(USABLE_MAP[iid])
        rec["split"] = split_name
        SELECTED_IMAGES.append(rec)

with open(ExperimentConfig.DATASET_3000_JSON, "w", encoding="utf-8") as f:
    json.dump({"images": SELECTED_IMAGES}, f, indent=2)
print("DATA SPLIT TEST PASSED: 2400 train / 300 val / 300 test pairwise disjoint.")

[MANIFEST VERIFIED] Reusing valid split manifest at C:\Users\LENOVO\Desktop\vansh\split_manifest.json
DATA SPLIT TEST PASSED: 2400 train / 300 val / 300 test pairwise disjoint.


In [7]:
# 09. Training-Only Vocabulary & Ground-Truth Label Files
train_only_records = [img for img in SELECTED_IMAGES if img["split"] == "train"]
assert len(train_only_records) == ExperimentConfig.TRAIN_COUNT

word_freq = Counter()
for img in train_only_records:
    for sent in img["sentences"]:
        word_freq.update(sent["tokens"])

vocab_words = sorted([w for w, c in word_freq.items() if c >= ExperimentConfig.MIN_WORD_FREQ])
if "UNK" not in vocab_words:
    vocab_words.append("UNK")
vocab_words.sort()

# 1-indexed vocabulary (0 is reserved for <bos>/<eos>/<pad> in HATNet)
ITOW = {str(i + 1): w for i, w in enumerate(vocab_words)}
WTOI = {w: i + 1 for i, w in enumerate(vocab_words)}
UNK_IDX = WTOI["UNK"]
VOCAB_SIZE = len(vocab_words)

# Verify zero vocabulary leakage from val/test-exclusive words
train_unique_Raw = set(word_freq.keys())
for w in vocab_words:
    if w != "UNK":
        assert w in train_unique_Raw and word_freq[w] >= ExperimentConfig.MIN_WORD_FREQ, f"Vocab leakage detected for {w}!"

H5_LABEL_PATH = os.path.join(ExperimentConfig.OUT_DATA_DIR, "rsicd_label.h5")
INFO_JSON_PATH = os.path.join(ExperimentConfig.OUT_DATA_DIR, "rsicd_info.json")

total_caps = sum(len(img["sentences"]) for img in SELECTED_IMAGES)
label_arrays = np.zeros((total_caps, ExperimentConfig.MAX_CAPTION_LEN), dtype="int32")
label_length = np.zeros(total_caps, dtype="int32")
label_start_ix = np.zeros(len(SELECTED_IMAGES), dtype="int32")
label_end_ix = np.zeros(len(SELECTED_IMAGES), dtype="int32")

seq_cursor = 0
for i, img in enumerate(SELECTED_IMAGES):
    label_start_ix[i] = seq_cursor + 1  # 1-indexed Lua convention used by dataloader.py
    for sent in img["sentences"]:
        toks = sent["tokens"][:ExperimentConfig.MAX_CAPTION_LEN]
        label_length[seq_cursor] = len(toks)
        for k, tok in enumerate(toks):
            label_arrays[seq_cursor, k] = WTOI.get(tok, UNK_IDX)
        seq_cursor += 1
    label_end_ix[i] = seq_cursor

# Reuse existing H5 label file if already written and valid to prevent Windows file-lock errors
need_write_h5 = True
if os.path.exists(H5_LABEL_PATH):
    try:
        with h5py.File(H5_LABEL_PATH, "r") as f:
            if "labels" in f and len(f["labels"]) == total_caps:
                need_write_h5 = False
    except Exception:
        need_write_h5 = True

if need_write_h5:
    try:
        if os.path.exists(H5_LABEL_PATH):
            os.remove(H5_LABEL_PATH)
    except Exception:
        pass
    with h5py.File(H5_LABEL_PATH, "w") as f:
        f.create_dataset("labels", data=label_arrays)
        f.create_dataset("label_start_ix", data=label_start_ix)
        f.create_dataset("label_end_ix", data=label_end_ix)
        f.create_dataset("label_length", data=label_length)

info_payload = {
    "ix_to_word": ITOW,
    "images": [{"id": int(img["imgid"]), "file_path": img["filename"], "split": img["split"]} for img in SELECTED_IMAGES]
}
with open(INFO_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(info_payload, f, indent=2)

def write_coco_reference_json(split_name: str, out_path: str):
    sub_imgs = [img for img in SELECTED_IMAGES if img["split"] == split_name]
    coco_obj = {"images": [], "annotations": [], "type": "captions", "licenses": [], "info": {}}
    ann_id = 1
    for img in sub_imgs:
        coco_obj["images"].append({"id": int(img["imgid"]), "file_name": img["filename"]})
        for sent in img["sentences"]:
            coco_obj["annotations"].append({
                "id": ann_id,
                "image_id": int(img["imgid"]),
                "caption": sent["raw"]
            })
            ann_id += 1
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(coco_obj, f)
    return out_path

TRAIN_COCO_JSON = write_coco_reference_json("train", os.path.join(ExperimentConfig.OUT_DATA_DIR, "rsicd_train.json"))
VAL_COCO_JSON = write_coco_reference_json("val", os.path.join(ExperimentConfig.OUT_DATA_DIR, "rsicd_val.json"))
TEST_COCO_JSON = write_coco_reference_json("test", os.path.join(ExperimentConfig.OUT_DATA_DIR, "rsicd_test.json"))
print(f"Train-only vocabulary built: {VOCAB_SIZE} words (UNK={UNK_IDX}). Zero val/test leakage verified.")


Train-only vocabulary built: 1241 words (UNK=1). Zero val/test leakage verified.


In [8]:
# 10. Official HATNet Source Acquisition
repo_captioning = os.path.join(ExperimentConfig.REPO_DIR, "captioning")
if not os.path.exists(repo_captioning):
    print(f"Cloning official hanhan68/HATNet repository into {ExperimentConfig.REPO_DIR}...")
    subprocess.run(["git", "clone", "https://github.com/hanhan68/HATNet.git", ExperimentConfig.REPO_DIR], check=True)
else:
    print(f"Official HATNet repository present at {ExperimentConfig.REPO_DIR}")

if ExperimentConfig.REPO_DIR not in sys.path:
    sys.path.insert(0, ExperimentConfig.REPO_DIR)


Official HATNet repository present at C:\Users\LENOVO\Desktop\vansh\HATNet


In [9]:
# 11. Compatibility Fixes for Official HATNet Source
import importlib

# 11.1 Lazy-loading captioning/models/__init__.py
models_init_file = os.path.join(ExperimentConfig.REPO_DIR, "captioning", "models", "__init__.py")
with open(models_init_file, "w", encoding="utf-8") as f:
    f.write('''from __future__ import absolute_import, division, print_function
def setup(opt):
    if opt.caption_model == 'transformer':
        from .GeoRSCLIP_mul_HAT_PTE import TransformerModel
        return TransformerModel(opt)
    raise Exception("Caption model not supported: {}".format(opt.caption_model))
''')
py_compile.compile(models_init_file, doraise=True)

transformer_bridge = os.path.join(ExperimentConfig.REPO_DIR, "captioning", "models", "TransformerModel.py")
if not os.path.exists(transformer_bridge):
    with open(transformer_bridge, "w", encoding="utf-8") as f:
        f.write("from .GeoRSCLIP_mul_HAT_PTE import subsequent_mask, TransformerModel, Generator\n")

# 11.2 Patch GeoRSCLIP_mul_HAT_PTE.py (MultiLevelAttEmbed & Registered DecoderLayer2 Attention)
pte_file = os.path.join(ExperimentConfig.REPO_DIR, "captioning", "models", "GeoRSCLIP_mul_HAT_PTE.py")
with open(pte_file, "r", encoding="utf-8") as f:
    pte_src = f.read().replace("\r\n", "\n")

multi_level_cls = '''
class MultiLevelAttEmbed(nn.Module):
    """Projects 3 concatenated GeoRSCLIP feature blocks (3 x 1280 = 3840) into 3 x d_model (1536)
    so EncoderDecoder.encode slices 0:512, 512:1024, 1024:1536 without dimension mismatch."""
    def __init__(self, att_feat_size, d_model, use_bn=0, drop_prob_lm=0.1):
        super(MultiLevelAttEmbed, self).__init__()
        self.d_model = d_model
        chunk_in = att_feat_size // 3 if att_feat_size > d_model * 3 else d_model
        def make_proj():
            layers = []
            if use_bn:
                layers.append(nn.BatchNorm1d(chunk_in))
            layers.append(nn.Linear(chunk_in, d_model))
            layers.append(nn.ReLU())
            if drop_prob_lm > 0:
                layers.append(nn.Dropout(drop_prob_lm))
            if use_bn == 2:
                layers.append(nn.BatchNorm1d(d_model))
            return nn.Sequential(*layers)
        self.proj1 = make_proj()
        self.proj2 = make_proj()
        self.proj3 = make_proj()

    def __getitem__(self, idx):
        return [self.proj1, self.proj2, self.proj3][idx]

    def __len__(self):
        return 3

    def forward(self, x):
        c = x.shape[-1] // 3
        return torch.cat([
            self.proj1(x[..., 0*c:1*c]),
            self.proj2(x[..., 1*c:2*c]),
            self.proj3(x[..., 2*c:3*c])
        ], dim=-1)
'''
if "class MultiLevelAttEmbed" not in pte_src:
    pte_src = pte_src.replace("class EncoderDecoder(nn.Module):", multi_level_cls + "\nclass EncoderDecoder(nn.Module):", 1)

if "self.yita = nn.Parameter" not in pte_src:
    pte_src, n_init = re.subn(
        r"(self\.fc_alpha2\s*=\s*nn\.Linear\(self\.d_model\s*\+\s*self\.d_model,\s*self\.d_model\))",
        r"\1\n        self.channel = ChannelAttention(self.d_model, self.d_model)\n        self.spatial = SpatialAttention()\n        self.yita = nn.Parameter(torch.tensor(0.2))",
        pte_src,
        count=1
    )
    assert n_init == 1, "Failed to register channel/spatial/yita in DecoderLayer2.__init__"

if "self.channel(Vm)" not in pte_src:
    fwd_pat = re.compile(
        r"channel\s*=\s*ChannelAttention\([^)]*\)\.to\([^)]*\)\s*\n"
        r"\s*spatial\s*=\s*SpatialAttention\(\)\.to\([^)]*\)\s*\n"
        r"\s*yita\s*=\s*nn\.Parameter\(.*?\)\.to\([^)]*\)\s*\n"
        r"\s*Vm\s*=\s*self\.mul_level_ffn\(m\)\s*\*\s*yita\s*\+\s*memory\[\.\.\.,\s*512\s*\*\s*2:512\s*\*\s*3\]\s*\n"
        r"\s*Vm\s*=\s*Vm\.view\(Vm\.shape\[0\],\s*Vm\.shape\[2\],\s*16,\s*16\).*?\n"
        r"\s*Vc\s*=\s*channel\(Vm\)\.view\(Vm\.shape\[0\],\s*-1,\s*Vm\.shape\[1\]\)\s*\n"
        r"\s*Vs\s*=\s*spatial\(Vm\)\.view\(Vm\.shape\[0\],\s*-1,\s*Vm\.shape\[1\]\)"
    )
    fwd_rep = (
        "Vm = self.mul_level_ffn(m) * self.yita + memory[..., self.d_model * 2:self.d_model * 3]\n"
        "        Vm = Vm.view(Vm.shape[0], Vm.shape[2], 16, 16)\n"
        "        Vc = self.channel(Vm).view(Vm.shape[0], -1, Vm.shape[1])\n"
        "        Vs = self.spatial(Vm).view(Vm.shape[0], -1, Vm.shape[1])"
    )
    pte_src, n_fwd = fwd_pat.subn(fwd_rep, pte_src, count=1)
    assert n_fwd == 1, "Failed to patch DecoderLayer2.forward"

if "MultiLevelAttEmbed(self.att_feat_size" not in pte_src:
    att_pat = re.compile(
        r"self\.att_embed\s*=\s*clones\([^)]*nn\.Sequential\(.*?\)\s*\)?,\s*3\)",
        re.DOTALL
    )
    pte_src, n_att = att_pat.subn(
        "self.att_embed = MultiLevelAttEmbed(self.att_feat_size, self.d_model, self.use_bn, self.drop_prob_lm)",
        pte_src,
        count=1
    )
    assert n_att == 1, f"Failed to patch self.att_embed (matched {n_att})"

if "nn.Identity()," not in pte_src:
    pte_src = pte_src.replace(
        "nn.Sequential(nn.Linear(d_model, d_model), nn.ReLU(), nn.Dropout(dropout)),",
        "nn.Identity(),"
    )

with open(pte_file, "w", encoding="utf-8") as f:
    f.write(pte_src)
py_compile.compile(pte_file, doraise=True)

# 11.3 Patch dataloader.py for safe h5py, loader_num_workers, & empty input_fc_dir
dl_file = os.path.join(ExperimentConfig.REPO_DIR, "captioning", "data", "dataloader.py")
with open(dl_file, "r", encoding="utf-8") as f:
    dl_src = f.read().replace("\r\n", "\n")
dl_src = re.sub(
    r"self\.h5_label_file\s*=\s*h5py\.File\(self\.opt\.input_label_h5,\s*'r'(?:,\s*driver='core')?\)",
    "self.h5_label_file = h5py.File(self.opt.input_label_h5, 'r')",
    dl_src
)
if "if not self.opt.input_fc_dir:" not in dl_src:
    dl_src = dl_src.replace(
        "return np.float32(self.fc_loader.get(str(self.info['images'][ix]['id'])))",
        "return np.zeros((1,), dtype='float32') if not self.opt.input_fc_dir else np.float32(self.fc_loader.get(str(self.info['images'][ix]['id'])))"
    )

# Fix Windows multiprocessing DataLoader pickling error by respecting loader_num_workers
dl_src = re.sub(
    r"num_workers\s*=\s*4[^\n]*\n\s*collate_fn\s*=\s*partial\([^)]*\)\s*,\s*\n\s*drop_last\s*=\s*False\s*,\s*\n\s*persistent_workers\s*=\s*True",
    "num_workers=getattr(self.opt, 'loader_num_workers', 0),\n                                                  collate_fn=partial(self.dataset.collate_func, split=split),\n                                                  drop_last=False,\n                                                  persistent_workers=(getattr(self.opt, 'loader_num_workers', 0) > 0)",
    dl_src
)
with open(dl_file, "w", encoding="utf-8") as f:
    f.write(dl_src)
py_compile.compile(dl_file, doraise=True)

# 11.4 Patch rewards.py to remove misleading warning
rewards_file = os.path.join(ExperimentConfig.REPO_DIR, "captioning", "utils", "rewards.py")
if os.path.exists(rewards_file):
    with open(rewards_file, "r", encoding="utf-8") as f:
        rew_src = f.read()
    if "print('cider or coco-caption missing')" in rew_src:
        rew_src = rew_src.replace("print('cider or coco-caption missing')", "pass")
        with open(rewards_file, "w", encoding="utf-8") as f:
            f.write(rew_src)
    py_compile.compile(rewards_file, doraise=True)

print("All official HATNet compatibility fixes applied and syntax-verified.")


All official HATNet compatibility fixes applied and syntax-verified.


In [10]:
# 12. GeoRSCLIP Checkpoint Verification
def resolve_and_verify_georsclip_ckpt(cand_ckpts):
    if not cand_ckpts:
        print("Downloading 'ckpt/RS5M_ViT-H-14.pt' from Hugging Face ('Zilun/GeoRSCLIP')...")
        dl_p = hf_hub_download(
            repo_id="Zilun/GeoRSCLIP",
            filename="ckpt/RS5M_ViT-H-14.pt",
            local_dir=ExperimentConfig.WORKSPACE_DIR,
            local_dir_use_symlinks=False
        )
        cand_ckpts = [os.path.abspath(dl_p)]
        
    for cp in cand_ckpts:
        actual_sha = compute_file_sha256(cp)
        if actual_sha.lower() == ExperimentConfig.EXPECTED_CKPT_SHA256.lower():
            print(f"[VERIFIED] GeoRSCLIP checkpoint: {cp}")
            print(f"[SHA-256 ] {actual_sha}")
            return cp, actual_sha
        elif ExperimentConfig.ALLOW_CKPT_HASH_OVERRIDE:
            print(f"[VERIFIED WITH OVERRIDE] Using checkpoint {cp} with SHA-256 {actual_sha}")
            return cp, actual_sha
        else:
            print(f"[REJECTED] Checkpoint {cp} hash {actual_sha} != {ExperimentConfig.EXPECTED_CKPT_SHA256}")
            
    raise RuntimeError("No GeoRSCLIP checkpoint matched expected SHA-256!")

RESOLVED_CKPT_PATH, VERIFIED_CKPT_SHA256 = resolve_and_verify_georsclip_ckpt(CAND_CKPTS)


[VERIFIED] GeoRSCLIP checkpoint: C:\Users\LENOVO\Desktop\vansh\ckpt\RS5M_ViT-H-14.pt
[SHA-256 ] 62a3e79df886875976901b4a9b8d6a1c42b457879a5ce625c8eba9460cbc3a11


In [11]:
# 13. Strict Feature-Cache Validation & Unit Test Suite (8 Mandatory Cache Tests)
class StrictFeatureCacheValidator:
    MANDATORY_KEYS = [
        "feat", "img_id", "filename", "dataset_fingerprint", "annotation_fingerprint",
        "checkpoint_sha256", "model_name", "layers", "preprocessing", "concat_order",
        "shape", "dtype", "cache_schema_version"
    ]

    @classmethod
    def validate_cache_file(
        cls,
        feat_file: str,
        expected_img_id: int,
        expected_filename: str,
        expected_dataset_fp: str,
        expected_ann_fp: str,
        expected_ckpt_sha256: str,
        expected_preprocess_str: str,
    ) -> tuple:
        if not os.path.exists(feat_file):
            return False, "file_missing"
        if feat_file.endswith(".tmp.npz") or os.path.exists(feat_file + ".tmp.npz"):
            return False, "interrupted_tmp_write_detected"
        try:
            with np.load(feat_file, allow_pickle=False) as data:
                for k in cls.MANDATORY_KEYS:
                    if k not in data:
                        return False, f"missing_metadata_key:{k}"
                if int(data["img_id"]) != int(expected_img_id):
                    return False, f"wrong_img_id:{int(data['img_id'])}!={int(expected_img_id)}"
                if str(data["filename"]) != str(expected_filename):
                    return False, "wrong_filename"
                if str(data["dataset_fingerprint"]) != str(expected_dataset_fp):
                    return False, "wrong_dataset_fingerprint"
                if str(data["annotation_fingerprint"]) != str(expected_ann_fp):
                    return False, "wrong_annotation_fingerprint"
                if str(data["checkpoint_sha256"]).lower() != str(expected_ckpt_sha256).lower():
                    return False, "wrong_checkpoint_sha256"
                if str(data["model_name"]) != ExperimentConfig.MODEL_NAME:
                    return False, "wrong_encoder_model_name"
                if list(data["layers"].tolist()) != list(ExperimentConfig.EXTRACTION_LAYERS):
                    return False, "wrong_extraction_layers"
                if str(data["preprocessing"]) != str(expected_preprocess_str):
                    return False, "changed_preprocessing"
                if str(data["concat_order"]) != ExperimentConfig.CONCAT_ORDER:
                    return False, "wrong_concat_order"
                if str(data["cache_schema_version"]) != ExperimentConfig.CACHE_SCHEMA_VERSION:
                    return False, "wrong_cache_schema_version"
                if tuple(data["shape"].tolist()) != (ExperimentConfig.NUM_PATCHES, ExperimentConfig.ATT_FEAT_SIZE):
                    return False, "wrong_metadata_shape"
                if str(data["dtype"]) != "float32":
                    return False, "wrong_metadata_dtype"
                    
                feat = data["feat"]
                if feat.shape != (ExperimentConfig.NUM_PATCHES, ExperimentConfig.ATT_FEAT_SIZE):
                    return False, f"wrong_tensor_shape:{feat.shape}"
                if feat.dtype != np.float32:
                    return False, f"wrong_tensor_dtype:{feat.dtype}"
                if not np.isfinite(feat).all() or float(np.abs(feat).sum()) == 0.0:
                    return False, "non_finite_or_zero_features"
                b1, b2, b3 = feat[:, :1280], feat[:, 1280:2560], feat[:, 2560:3840]
                if np.allclose(b1, b2) or np.allclose(b2, b3):
                    return False, "duplicated_layer_blocks"
            return True, "ok"
        except Exception as e:
            return False, f"corrupted_cache:{type(e).__name__}"

    @classmethod
    def save_atomic_cache(
        cls,
        feat_file: str,
        feat_np: np.ndarray,
        img_id: int,
        filename: str,
        dataset_fp: str,
        ann_fp: str,
        ckpt_sha256: str,
        preprocess_str: str,
    ):
        tmp_file = feat_file + ".tmp.npz"
        np.savez_compressed(
            tmp_file,
            feat=feat_np.astype(np.float32),
            img_id=np.int64(img_id),
            filename=np.str_(filename),
            dataset_fingerprint=np.str_(dataset_fp),
            annotation_fingerprint=np.str_(ann_fp),
            checkpoint_sha256=np.str_(ckpt_sha256),
            model_name=np.str_(ExperimentConfig.MODEL_NAME),
            layers=np.array(ExperimentConfig.EXTRACTION_LAYERS, dtype=np.int32),
            preprocessing=np.str_(preprocess_str),
            concat_order=np.str_(ExperimentConfig.CONCAT_ORDER),
            shape=np.array(feat_np.shape, dtype=np.int32),
            dtype=np.str_("float32"),
            cache_schema_version=np.str_(ExperimentConfig.CACHE_SCHEMA_VERSION),
        )
        os.replace(tmp_file, feat_file)

def run_8_mandatory_cache_tests():
    test_dir = os.path.join(ExperimentConfig.WORKSPACE_DIR, "_cache_unit_tests")
    os.makedirs(test_dir, exist_ok=True)
    p = os.path.join(test_dir, "99999.npz")
    valid_feat = np.random.randn(256, 3840).astype(np.float32)
    
    # 1. Valid cache acceptance
    StrictFeatureCacheValidator.save_atomic_cache(p, valid_feat, 99999, "test.jpg", "dfp", "afp", "sha_ok", "prep_v1")
    ok, reason = StrictFeatureCacheValidator.validate_cache_file(p, 99999, "test.jpg", "dfp", "afp", "sha_ok", "prep_v1")
    assert ok and reason == "ok", f"Test 1 failed: {reason}"
    
    # 2. Wrong image ID
    ok, reason = StrictFeatureCacheValidator.validate_cache_file(p, 11111, "test.jpg", "dfp", "afp", "sha_ok", "prep_v1")
    assert not ok and "wrong_img_id" in reason, f"Test 2 failed: {reason}"
    
    # 3. Wrong checkpoint hash
    ok, reason = StrictFeatureCacheValidator.validate_cache_file(p, 99999, "test.jpg", "dfp", "afp", "bad_sha", "prep_v1")
    assert not ok and "wrong_checkpoint_sha256" in reason, f"Test 3 failed: {reason}"
    
    # 4. Changed preprocessing
    ok, reason = StrictFeatureCacheValidator.validate_cache_file(p, 99999, "test.jpg", "dfp", "afp", "sha_ok", "prep_v2")
    assert not ok and "changed_preprocessing" in reason, f"Test 4 failed: {reason}"
    
    # 5. Wrong feature shape
    bad_shape_p = os.path.join(test_dir, "bad_shape.npz")
    StrictFeatureCacheValidator.save_atomic_cache(bad_shape_p, np.random.randn(196, 3840).astype(np.float32), 99999, "test.jpg", "dfp", "afp", "sha_ok", "prep_v1")
    ok, reason = StrictFeatureCacheValidator.validate_cache_file(bad_shape_p, 99999, "test.jpg", "dfp", "afp", "sha_ok", "prep_v1")
    assert not ok and "shape" in reason, f"Test 5 failed: {reason}"
    
    # 6. Corrupted cache
    corrupt_p = os.path.join(test_dir, "corrupt.npz")
    with open(corrupt_p, "wb") as f:
        f.write(b"corrupted_binary_header_bytes")
    ok, reason = StrictFeatureCacheValidator.validate_cache_file(corrupt_p, 99999, "test.jpg", "dfp", "afp", "sha_ok", "prep_v1")
    assert not ok and "corrupted_cache" in reason, f"Test 6 failed: {reason}"
    
    # 7. Missing metadata
    missing_meta_p = os.path.join(test_dir, "missing_meta.npz")
    np.savez(missing_meta_p, feat=valid_feat)
    ok, reason = StrictFeatureCacheValidator.validate_cache_file(missing_meta_p, 99999, "test.jpg", "dfp", "afp", "sha_ok", "prep_v1")
    assert not ok and "missing_metadata_key" in reason, f"Test 7 failed: {reason}"
    
    # 8. Interrupted cache write (.tmp.npz present)
    with open(p + ".tmp.npz", "wb") as f:
        f.write(b"partial")
    ok, reason = StrictFeatureCacheValidator.validate_cache_file(p, 99999, "test.jpg", "dfp", "afp", "sha_ok", "prep_v1")
    assert not ok and "interrupted_tmp_write_detected" in reason, f"Test 8 failed: {reason}"
    
    shutil.rmtree(test_dir, ignore_errors=True)
    print("ALL 8 MANDATORY FEATURE-CACHE VALIDATION TESTS PASSED.")

run_8_mandatory_cache_tests()


ALL 8 MANDATORY FEATURE-CACHE VALIDATION TESTS PASSED.


In [12]:
# 14. Batched GeoRSCLIP Feature Extraction & Exhaustive Cache Verification
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms("ViT-H-14", pretrained=RESOLVED_CKPT_PATH)
clip_model = clip_model.to(DEVICE).eval()
for param in clip_model.parameters():
    param.requires_grad = False

PREPROCESS_STR = str(clip_preprocess)

# Verify which of the 3,000 images need extraction
to_extract = []
cached_valid = 0
rejection_reasons = Counter()

for img_meta in SELECTED_IMAGES:
    iid = int(img_meta["imgid"])
    fname = img_meta["filename"]
    feat_p = os.path.join(ExperimentConfig.FEATURES_DIR, f"{iid}.npz")
    ok, reason = StrictFeatureCacheValidator.validate_cache_file(
        feat_p, iid, fname, DATASET_FINGERPRINT, ANNOTATION_FINGERPRINT, VERIFIED_CKPT_SHA256, PREPROCESS_STR
    )
    if ok:
        cached_valid += 1
    else:
        if reason != "file_missing":
            rejection_reasons[reason] += 1
            if os.path.exists(feat_p + ".tmp.npz"):
                os.remove(feat_p + ".tmp.npz")
        to_extract.append(img_meta)

if rejection_reasons:
    print(f"Logged invalid cache rejections: {dict(rejection_reasons)}")
print(f"Feature cache status: {cached_valid} valid cached | {len(to_extract)} to extract.")

# Batched extraction to maximize GPU throughput without loading all images into RAM
bs = ExperimentConfig.EXTRACT_BATCH_SIZE
target_layers = set(ExperimentConfig.EXTRACTION_LAYERS)

for start_idx in tqdm(range(0, len(to_extract), bs), desc="Extracting GeoRSCLIP ViT-H/14 features"):
    batch_metas = to_extract[start_idx : start_idx + bs]
    pil_imgs = []
    for m in batch_metas:
        base_fn = os.path.basename(m["filename"])
        if base_fn in HF_IMAGE_MAP:
            img = HF_DATASET[HF_IMAGE_MAP[base_fn]]["image"].convert("RGB")
        else:
            img = Image.open(os.path.join(RESOLVED_IMG_DIR, m["filename"])).convert("RGB")
        pil_imgs.append(clip_preprocess(img))
        
    batch_tensor = torch.stack(pil_imgs, dim=0).to(DEVICE)
    with torch.no_grad():
        x = clip_model.visual.conv1(batch_tensor)
        x = x.reshape(x.shape[0], x.shape[1], -1).permute(0, 2, 1)
        cls_tok = clip_model.visual.class_embedding.to(x.dtype) + torch.zeros(
            x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device
        )
        x = torch.cat([cls_tok, x], dim=1)
        x = x + clip_model.visual.positional_embedding.to(x.dtype)
        x = clip_model.visual.ln_pre(x)
        
        batch_first = getattr(clip_model.visual.transformer, "batch_first", True)
        if not batch_first:
            x = x.transpose(0, 1).contiguous()
            
        layer_outputs = []
        for blk_idx, blk in enumerate(clip_model.visual.transformer.resblocks):
            x = blk(x)
            if blk_idx in target_layers:
                patch_tokens = x[:, 1:, :].clone() if batch_first else x[1:, :, :].transpose(0, 1).clone()
                layer_outputs.append(patch_tokens)
                
        concat_batch = torch.cat(layer_outputs, dim=-1).cpu().numpy().astype(np.float32)  # (B, 256, 3840)
        
    for b_i, m in enumerate(batch_metas):
        iid = int(m["imgid"])
        feat_p = os.path.join(ExperimentConfig.FEATURES_DIR, f"{iid}.npz")
        StrictFeatureCacheValidator.save_atomic_cache(
            feat_p, concat_batch[b_i], iid, m["filename"],
            DATASET_FINGERPRINT, ANNOTATION_FINGERPRINT, VERIFIED_CKPT_SHA256, PREPROCESS_STR
        )

del clip_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Save feature cache metadata manifest
feature_cache_meta = {
    "total_verified_files": len(SELECTED_IMAGES),
    "model_name": ExperimentConfig.MODEL_NAME,
    "checkpoint_sha256": VERIFIED_CKPT_SHA256,
    "extraction_layers": ExperimentConfig.EXTRACTION_LAYERS,
    "concat_order": ExperimentConfig.CONCAT_ORDER,
    "feature_shape": [ExperimentConfig.NUM_PATCHES, ExperimentConfig.ATT_FEAT_SIZE],
    "dtype": "float32",
    "dataset_fingerprint": DATASET_FINGERPRINT,
    "annotation_fingerprint": ANNOTATION_FINGERPRINT,
    "cache_schema_version": ExperimentConfig.CACHE_SCHEMA_VERSION
}
with open(ExperimentConfig.FEATURE_CACHE_META_JSON, "w", encoding="utf-8") as f:
    json.dump(feature_cache_meta, f, indent=2)
print(f"Feature extraction complete and metadata saved to {ExperimentConfig.FEATURE_CACHE_META_JSON}")

Feature cache status: 3000 valid cached | 0 to extract.


Extracting GeoRSCLIP ViT-H/14 features: 0it [00:00, ?it/s]

Feature extraction complete and metadata saved to C:\Users\LENOVO\Desktop\vansh\feature_cache_metadata.json


In [13]:
# 15. Training-Only Retrieval Memory & Leakage Prevention Safeguards
class TrainingOnlyRetrievalMemory:
    """Constructs an image-caption retrieval memory exclusively from the 2,400 training images.
    Enforces strict self-exclusion during training and forbids any val/test IDs from entering memory."""

    def __init__(
        self,
        train_records: list,
        val_ids_set: set,
        test_ids_set: set,
        features_dir: str,
        wtoi: dict,
        unk_idx: int,
        max_caption_len: int = 20,
        memory_mode: str = "block31_mean_norm",
    ):
        self.val_ids_set = set(int(x) for x in val_ids_set)
        self.test_ids_set = set(int(x) for x in test_ids_set)
        self.features_dir = features_dir
        self.wtoi = wtoi
        self.unk_idx = unk_idx
        self.max_caption_len = max_caption_len
        self.memory_mode = memory_mode
        
        self.train_ids = []
        self.id_to_index = {}
        self.captions_by_id = {}
        self.token_ids_by_index = []   # List of (5, max_caption_len) int64 arrays
        self.token_masks_by_index = [] # List of (5, max_caption_len) float32 arrays
        embeddings = []
        
        for idx, rec in enumerate(train_records):
            iid = int(rec["imgid"])
            assert iid not in self.val_ids_set, f"LEAKAGE ERROR: Validation ID {iid} in training memory!"
            assert iid not in self.test_ids_set, f"LEAKAGE ERROR: Test ID {iid} in training memory!"
            assert rec.get("split", "train") == "train", f"Non-train split record {iid} passed to memory!"
            
            self.train_ids.append(iid)
            self.id_to_index[iid] = idx
            self.captions_by_id[iid] = list(rec["raw_captions"])
            
            # Encode up to 5 reference captions into padded token matrices
            cap_tok_mat = np.zeros((5, max_caption_len), dtype=np.int64)
            cap_msk_mat = np.zeros((5, max_caption_len), dtype=np.float32)
            sents = rec["sentences"][:5]
            for c_i in range(5):
                sent = sents[c_i % len(sents)]
                toks = sent["tokens"][:max_caption_len]
                for t_i, tok in enumerate(toks):
                    cap_tok_mat[c_i, t_i] = wtoi.get(tok, unk_idx)
                    cap_msk_mat[c_i, t_i] = 1.0
            self.token_ids_by_index.append(cap_tok_mat)
            self.token_masks_by_index.append(cap_msk_mat)
            
            # Extract image embedding from cached GeoRSCLIP features
            feat_p = os.path.join(features_dir, f"{iid}.npz")
            with np.load(feat_p) as d:
                feat = d["feat"]  # (256, 3840)
            emb = self.compute_query_embedding_np(feat[None, ...])[0]
            embeddings.append(emb)
            
        self.train_ids_set = set(self.train_ids)
        self.memory_embeddings = torch.from_numpy(np.stack(embeddings, axis=0)).float()  # (2400, D_emb)
        self.memory_token_ids = torch.from_numpy(np.stack(self.token_ids_by_index, axis=0)).long()     # (2400, 5, L)
        self.memory_token_masks = torch.from_numpy(np.stack(self.token_masks_by_index, axis=0)).float() # (2400, 5, L)

    def compute_query_embedding_np(self, att_feats_np: np.ndarray) -> np.ndarray:
        if self.memory_mode == "block31_mean_norm":
            raw_emb = att_feats_np[:, :, 2560:3840].mean(axis=1)
        else:
            raw_emb = att_feats_np.mean(axis=1)
        norms = np.linalg.norm(raw_emb, axis=-1, keepdims=True) + 1e-8
        return (raw_emb / norms).astype(np.float32)

    def compute_query_embedding_torch(self, att_feats: torch.Tensor) -> torch.Tensor:
        if self.memory_mode == "block31_mean_norm":
            raw_emb = att_feats[:, :, 2560:3840].mean(dim=1)
        else:
            raw_emb = att_feats.mean(dim=1)
        return F.normalize(raw_emb, p=2, dim=-1)

    def retrieve_batch(
        self,
        query_att_feats: torch.Tensor,
        query_image_ids: list = None,
        is_training: bool = False,
        top_k: int = None,
        temperature: float = None,
        score_threshold: float = None,
    ) -> dict:
        """Retrieves top-K training examples per query image in `query_att_feats` (B, 256, 3840).
        Explicitly masks out self-matches when `is_training=True` or when `query_image_ids` is in memory."""
        k = top_k or ExperimentConfig.RETRIEVAL_TOP_K
        temp = temperature or ExperimentConfig.RETRIEVAL_TEMPERATURE
        thresh = score_threshold if score_threshold is not None else ExperimentConfig.RETRIEVAL_SCORE_THRESHOLD
        
        device = query_att_feats.device
        mem_emb = self.memory_embeddings.to(device)  # (N_mem, D)
        q_emb = self.compute_query_embedding_torch(query_att_feats)  # (B, D)
        
        sim_matrix = torch.matmul(q_emb, mem_emb.t())  # (B, N_mem)
        
        if is_training and query_image_ids is None:
            raise ValueError("query_image_ids must be provided during training to enforce self-match exclusion!")
            
        if query_image_ids is not None:
            assert len(query_image_ids) == query_att_feats.shape[0]
            for b_i, q_id in enumerate(query_image_ids):
                q_id_int = int(q_id)
                if q_id_int in self.id_to_index:
                    mem_idx = self.id_to_index[q_id_int]
                    sim_matrix[b_i, mem_idx] = -1e9  # Strict self-exclusion
                    
        top_scores, top_indices = torch.topk(sim_matrix, k=k, dim=-1, largest=True, sorted=True)  # (B, K)
        
        # Apply threshold mask if configured
        valid_mask = (top_scores >= thresh).float()
        # Ensure at least top-1 remains active as documented fallback if all fall below threshold
        valid_mask[:, 0] = 1.0
        
        masked_logits = (top_scores / max(temp, 1e-4)) + (1.0 - valid_mask) * (-1e9)
        sim_weights = F.softmax(masked_logits, dim=-1)  # (B, K)
        
        cpu_indices = top_indices.detach().cpu()
        ret_tok_ids = self.memory_token_ids[cpu_indices].to(device)     # (B, K, 5, L)
        ret_tok_masks = self.memory_token_masks[cpu_indices].to(device) # (B, K, 5, L)
        
        retrieved_img_ids = [
            [self.train_ids[idx_val] for idx_val in row.tolist()]
            for row in cpu_indices
        ]
        
        # Runtime leakage assertions
        for b_i, r_ids in enumerate(retrieved_img_ids):
            for r_id in r_ids:
                assert r_id in self.train_ids_set, f"Retrieved non-training ID {r_id}!"
                assert r_id not in self.val_ids_set and r_id not in self.test_ids_set
                if query_image_ids is not None:
                    assert int(r_id) != int(query_image_ids[b_i]), (
                        f"SELF-MATCH LEAKAGE: Query {query_image_ids[b_i]} retrieved itself!"
                    )
                    
        return {
            "retrieved_ids": retrieved_img_ids,
            "top_scores": top_scores,
            "sim_weights": sim_weights,
            "token_ids": ret_tok_ids,
            "token_masks": ret_tok_masks,
            "valid_mask": valid_mask,
        }

RETRIEVAL_MEMORY = TrainingOnlyRetrievalMemory(
    train_records=train_only_records,
    val_ids_set=VAL_IDS_SET,
    test_ids_set=TEST_IDS_SET,
    features_dir=ExperimentConfig.FEATURES_DIR,
    wtoi=WTOI,
    unk_idx=UNK_IDX,
    max_caption_len=ExperimentConfig.MAX_CAPTION_LEN,
    memory_mode=ExperimentConfig.RETRIEVAL_MEMORY_MODE,
)

retrieval_meta_payload = {
    "memory_size": len(RETRIEVAL_MEMORY.train_ids),
    "memory_mode": ExperimentConfig.RETRIEVAL_MEMORY_MODE,
    "top_k": ExperimentConfig.RETRIEVAL_TOP_K,
    "similarity": ExperimentConfig.RETRIEVAL_SIMILARITY,
    "temperature": ExperimentConfig.RETRIEVAL_TEMPERATURE,
    "score_threshold": ExperimentConfig.RETRIEVAL_SCORE_THRESHOLD,
    "train_ids_sha256": hashlib.sha256(
        ",".join(str(x) for x in RETRIEVAL_MEMORY.train_ids).encode("utf-8")
    ).hexdigest(),
    "contains_val_or_test": False
}
RETRIEVAL_MEMORY.metadata = retrieval_meta_payload
with open(ExperimentConfig.RETRIEVAL_META_JSON, "w", encoding="utf-8") as f:
    json.dump(retrieval_meta_payload, f, indent=2)
print(f"Training-only retrieval memory built ({len(RETRIEVAL_MEMORY.train_ids)} images) and saved metadata.")


Training-only retrieval memory built (2400 images) and saved metadata.


In [14]:
# 16. Retrieval Leakage & Quality Verification Tests
def run_retrieval_leakage_and_quality_tests(memory: TrainingOnlyRetrievalMemory):
    # 1. Verify memory contains strictly training IDs
    assert len(memory.train_ids) == ExperimentConfig.TRAIN_COUNT
    assert memory.train_ids_set.isdisjoint(VAL_IDS_SET)
    assert memory.train_ids_set.isdisjoint(TEST_IDS_SET)
    
    # 2. Deliberate self-match exclusion test on first 4 training images
    sample_train_ids = memory.train_ids[:4]
    sample_feats = []
    for iid in sample_train_ids:
        with np.load(os.path.join(ExperimentConfig.FEATURES_DIR, f"{iid}.npz")) as d:
            sample_feats.append(d["feat"])
    q_tensor = torch.from_numpy(np.stack(sample_feats, axis=0)).to(DEVICE)
    
    ret_train = memory.retrieve_batch(q_tensor, query_image_ids=sample_train_ids, is_training=True, top_k=5)
    for b_i, q_id in enumerate(sample_train_ids):
        assert q_id not in ret_train["retrieved_ids"][b_i], f"Self-match exclusion failed for {q_id}!"
        # Verify descending similarity ordering
        scores = ret_train["top_scores"][b_i].tolist()
        assert scores == sorted(scores, reverse=True), "Retrieved scores are not sorted descending!"
        
    # 3. Verify that calling training retrieval without query_image_ids raises ValueError
    try:
        memory.retrieve_batch(q_tensor, query_image_ids=None, is_training=True)
        raise AssertionError("Expected ValueError when query_image_ids is None in training mode!")
    except ValueError:
        pass
        
    # 4. Evaluate retrieval semantic overlap (token IoU) on held-out validation subset
    val_sample_ids = SPLIT_MANIFEST["val_ids"][:20]
    val_feats = []
    for vid in val_sample_ids:
        with np.load(os.path.join(ExperimentConfig.FEATURES_DIR, f"{vid}.npz")) as d:
            val_feats.append(d["feat"])
    val_q = torch.from_numpy(np.stack(val_feats, axis=0)).to(DEVICE)
    ret_val = memory.retrieve_batch(val_q, query_image_ids=val_sample_ids, is_training=False, top_k=4)
    
    content_overlaps = []
    stop_words = {"a", "an", "the", "of", "in", "on", "is", "are", "with", "and", "near", "by", "to", "there"}
    for b_i, vid in enumerate(val_sample_ids):
        gt_words = set(
            w for cap in SPLIT_MANIFEST["val_captions"][str(vid)]
            for w in cap.split() if w not in stop_words
        )
        ret_words = set(
            w for rid in ret_val["retrieved_ids"][b_i]
            for cap in memory.captions_by_id[rid]
            for w in cap.split() if w not in stop_words
        )
        if gt_words:
            content_overlaps.append(len(gt_words & ret_words) / len(gt_words))
    mean_recall = float(np.mean(content_overlaps)) if content_overlaps else 0.0
    print(f"RETRIEVAL LEAKAGE & QUALITY TESTS PASSED (Held-out Val Content-Word Recall@4 = {mean_recall*100:.1f}%).")

run_retrieval_leakage_and_quality_tests(RETRIEVAL_MEMORY)

RETRIEVAL LEAKAGE & QUALITY TESTS PASSED (Held-out Val Content-Word Recall@4 = 65.4%).


In [15]:
# 17. Retrieved-Caption Encoding, ViPCap-Inspired Visual Prompt Mapper, and Prompt Fusion
class RetrievedCaptionEncoder(nn.Module):
    """Component F: Converts retrieved token IDs (B, K, 5, L), masks (B, K, 5, L), and
    similarity weights (B, K) into:
      1. `weighted_summary`: (B, d_model) similarity-weighted semantic vector.
      2. `retrieved_token_bank`: (B, K * L, d_model) similarity-scaled token states for cross-attention.
      3. `retrieved_bank_mask`: (B, K * L) validity mask."""

    def __init__(self, vocab_size: int, d_model: int = 512, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.token_embed = nn.Embedding(vocab_size + 1, d_model, padding_idx=0)
        self.pos_embed = nn.Embedding(64, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.sent_proj = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model)
        )

    def forward(self, token_ids: torch.Tensor, token_masks: torch.Tensor, sim_weights: torch.Tensor) -> dict:
        # token_ids: (B, K, 5, L), token_masks: (B, K, 5, L), sim_weights: (B, K)
        B, K, M, L = token_ids.shape
        pos_idx = torch.arange(L, device=token_ids.device).view(1, 1, 1, L)
        emb = self.token_embed(token_ids) + self.pos_embed(pos_idx)  # (B, K, M, L, D)
        emb = self.dropout(self.norm(emb))
        
        msk = token_masks.unsqueeze(-1)  # (B, K, M, L, 1)
        # Mean-pool over the M=5 captions per retrieved image at each token position, or per-caption summary
        cap_lengths = msk.sum(dim=3).clamp(min=1.0)  # (B, K, M, 1)
        per_cap_vec = (emb * msk).sum(dim=3) / cap_lengths  # (B, K, M, D)
        per_neighbor_vec = self.sent_proj(per_cap_vec.mean(dim=2))  # (B, K, D)
        
        # Similarity-weighted summary across top-K neighbors
        w = sim_weights.unsqueeze(-1)  # (B, K, 1)
        weighted_summary = (per_neighbor_vec * w).sum(dim=1)  # (B, D)
        
        # Representative token sequence per neighbor (averaging the 5 caption embeddings at token level
        # concatenated with the neighbor's sentence summary token) -> (B, K * (L + 1), D)
        primary_tok_emb = emb[:, :, 0, :, :] * (w.unsqueeze(-1))  # (B, K, L, D) scaled by similarity
        primary_tok_msk = token_masks[:, :, 0, :]                 # (B, K, L)
        
        bank_tokens = torch.cat([per_neighbor_vec.unsqueeze(2), primary_tok_emb], dim=2)  # (B, K, L+1, D)
        bank_masks = torch.cat([
            torch.ones(B, K, 1, device=token_ids.device, dtype=primary_tok_msk.dtype),
            primary_tok_msk
        ], dim=2)  # (B, K, L+1)
        
        return {
            "weighted_summary": weighted_summary,
            "neighbor_vectors": per_neighbor_vec,
            "token_bank": bank_tokens.view(B, K * (L + 1), self.d_model),
            "bank_mask": bank_masks.view(B, K * (L + 1)),
        }


class ViPCapPromptMapper(nn.Module):
    """Component G: ViPCap-Inspired Visual Prompt Mapper.
    Note on ViPCap Distinction: Published ViPCap uses a CLIP-conditioned Gaussian/MLP sampler over
    retrieved concepts. Our adaptation uses `num_prompt_tokens` learnable query vectors that
    cross-attend to similarity-weighted retrieved caption tokens and are conditioned on the
    weighted retrieval summary via a residual MLP to output visual-prompt tokens (B, N_p, d_model)."""

    def __init__(self, d_model: int = 512, num_prompt_tokens: int = 16, num_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.num_prompt_tokens = num_prompt_tokens
        self.prompt_queries = nn.Parameter(torch.randn(1, num_prompt_tokens, d_model) * 0.02)
        self.cross_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.summary_cond = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, encoded_retrieval: dict) -> torch.Tensor:
        token_bank = encoded_retrieval["token_bank"]          # (B, S, D)
        bank_mask = encoded_retrieval["bank_mask"]            # (B, S), 1=valid, 0=pad
        summary = encoded_retrieval["weighted_summary"]       # (B, D)
        B = token_bank.shape[0]
        
        queries = self.prompt_queries.expand(B, -1, -1)       # (B, N_p, D)
        key_padding_mask = (bank_mask <= 0.0)                 # True where padded
        
        attn_out, _ = self.cross_attn(
            query=queries,
            key=token_bank,
            value=token_bank,
            key_padding_mask=key_padding_mask
        )
        p = self.norm1(queries + attn_out)
        summary_exp = summary.unsqueeze(1).expand(-1, self.num_prompt_tokens, -1)
        p = self.norm2(p + self.summary_cond(torch.cat([p, summary_exp], dim=-1)))
        return p  # (B, N_p, D)


class VisualPromptFusion(nn.Module):
    """Component H: Integrates either Model B (retrieval_only summary) or Model C (full_vipcap prompt tokens)
    into HATNet's high-level visual memory slice M^{(3)} = memory[:, :, 2*d_model : 3*d_model] (B, 256, 512)
    right before Decoder2's ChannelAttention and SpatialAttention."""

    def __init__(self, d_model: int = 512, num_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        # Model C cross-attention between 256 visual patches and N_p visual prompt tokens
        self.visual_prompt_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.gate_proj = nn.Linear(d_model * 2, d_model)
        # Model B simple linear projection for retrieval_only ablation
        self.retrieval_only_proj = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.Tanh()
        )
        self.norm = nn.LayerNorm(d_model)
        # Zero-initialized scalar residual scale so initial forward pass matches baseline
        self.fusion_alpha = nn.Parameter(torch.tensor(0.1))

    def forward(
        self,
        memory: torch.Tensor,
        prompt_tokens: torch.Tensor = None,
        retrieval_summary: torch.Tensor = None,
        ab_mode: str = "full_vipcap",
    ) -> torch.Tensor:
        if memory.ndim != 3 or memory.shape[-1] != self.d_model * 3:
            raise ValueError(f"Expected HATNet memory shape (B, 256, {self.d_model*3}), got {tuple(memory.shape)}")
            
        if ab_mode == "baseline" or (prompt_tokens is None and retrieval_summary is None):
            return memory
            
        m1 = memory[..., 0*self.d_model : 1*self.d_model]
        m2 = memory[..., 1*self.d_model : 2*self.d_model]
        m3 = memory[..., 2*self.d_model : 3*self.d_model]  # (B, 256, 512)
        
        if ab_mode == "retrieval_only":
            if retrieval_summary is None:
                raise ValueError("retrieval_summary required for ab_mode='retrieval_only'")
            cond = self.retrieval_only_proj(retrieval_summary).unsqueeze(1).expand_as(m3)
            gate = torch.sigmoid(self.gate_proj(torch.cat([m3, cond], dim=-1)))
            m3_enhanced = self.norm(m3 + torch.tanh(self.fusion_alpha) * (gate * cond))
        elif ab_mode == "full_vipcap":
            if prompt_tokens is None:
                raise ValueError("prompt_tokens required for ab_mode='full_vipcap'")
            if prompt_tokens.ndim != 3 or prompt_tokens.shape[-1] != self.d_model:
                raise ValueError(f"Invalid prompt_tokens shape: {tuple(prompt_tokens.shape)}")
            attn_delta, _ = self.visual_prompt_attn(query=m3, key=prompt_tokens, value=prompt_tokens)
            gate = torch.sigmoid(self.gate_proj(torch.cat([m3, attn_delta], dim=-1)))
            m3_enhanced = self.norm(m3 + torch.tanh(self.fusion_alpha) * (gate * attn_delta))
        else:
            raise ValueError(f"Unrecognized ab_mode: {ab_mode}")
            
        return torch.cat([m1, m2, m3_enhanced], dim=-1)

In [16]:
# 18. Unified Retrieval-Enhanced & Baseline HATNet Wrapper
from captioning.models import setup as setup_base_hatnet

class RetrievalEnhancedHATNet(nn.Module):
    """Wraps the official HATNet `TransformerModel` (`GeoRSCLIP_mul_HAT_PTE.py`).
    - When `use_retrieval=False` or `ab_mode='baseline'` (Model A): Runs the exact unmodified
      HATNet baseline path and asserts that no retrieval data is used.
    - When `use_retrieval=True` and `ab_mode='retrieval_only'` (Model B): Uses RetrievedCaptionEncoder
      + gated linear conditioning.
    - When `use_retrieval=True` and `ab_mode='full_vipcap'` (Model C): Uses RetrievedCaptionEncoder
      + ViPCapPromptMapper + VisualPromptFusion before Decoder2 hierarchical attention."""

    def __init__(
        self,
        opt,
        use_retrieval: bool = False,
        ab_mode: str = "baseline",
        num_prompt_tokens: int = 16,
    ):
        super().__init__()
        self.opt = copy.deepcopy(opt)
        self.use_retrieval = bool(use_retrieval and ab_mode != "baseline")
        self.ab_mode = "baseline" if not self.use_retrieval else ab_mode
        self.d_model = opt.d_model
        
        # Instantiate official HATNet backbone
        self.hatnet = setup_base_hatnet(self.opt)
        
        if self.use_retrieval:
            self.caption_encoder = RetrievedCaptionEncoder(
                vocab_size=self.opt.vocab_size,
                d_model=self.d_model,
                dropout=ExperimentConfig.PROMPT_DROPOUT,
            )
            if self.ab_mode == "full_vipcap":
                self.prompt_mapper = ViPCapPromptMapper(
                    d_model=self.d_model,
                    num_prompt_tokens=num_prompt_tokens,
                    num_heads=self.opt.num_att_heads,
                    dropout=ExperimentConfig.PROMPT_DROPOUT,
                )
            else:
                self.prompt_mapper = None
            self.prompt_fusion = VisualPromptFusion(
                d_model=self.d_model,
                num_heads=self.opt.num_att_heads,
                dropout=ExperimentConfig.PROMPT_DROPOUT,
            )
        else:
            self.caption_encoder = None
            self.prompt_mapper = None
            self.prompt_fusion = None
            
        # Bind custom encode hook onto self.hatnet.model (EncoderDecoder)
        self._active_retrieval_context = None
        orig_encode = self.hatnet.model.encode
        
        def patched_encode(src, src_mask):
            memory = orig_encode(src, src_mask)
            if not self.use_retrieval or self._active_retrieval_context is None:
                return memory
            ctx = self._active_retrieval_context
            B_mem = memory.shape[0]
            prompt_toks = ctx.get("prompt_tokens")
            ret_sum = ctx.get("retrieval_summary")
            
            # Handle expansion when seq_per_img=5 during training or beam_size during decoding
            if prompt_toks is not None and prompt_toks.shape[0] != B_mem:
                repeat_factor = B_mem // prompt_toks.shape[0]
                prompt_toks = prompt_toks.unsqueeze(1).repeat(1, repeat_factor, 1, 1).view(B_mem, -1, self.d_model)
            if ret_sum is not None and ret_sum.shape[0] != B_mem:
                repeat_factor = B_mem // ret_sum.shape[0]
                ret_sum = ret_sum.unsqueeze(1).repeat(1, repeat_factor, 1).view(B_mem, self.d_model)
                
            return self.prompt_fusion(
                memory,
                prompt_tokens=prompt_toks,
                retrieval_summary=ret_sum,
                ab_mode=self.ab_mode,
            )
            
        self.hatnet.model.encode = patched_encode

    def _prepare_retrieval_context(self, retrieval_batch: dict):
        if not self.use_retrieval:
            if retrieval_batch is not None:
                raise RuntimeError("Baseline model (use_retrieval=False) must not receive retrieval_batch!")
            return None
        if retrieval_batch is None:
            raise ValueError("Retrieval-enhanced model requires retrieval_batch!")
        enc = self.caption_encoder(
            token_ids=retrieval_batch["token_ids"],
            token_masks=retrieval_batch["token_masks"],
            sim_weights=retrieval_batch["sim_weights"],
        )
        p_toks = self.prompt_mapper(enc) if self.prompt_mapper is not None else None
        return {
            "prompt_tokens": p_toks,
            "retrieval_summary": enc["weighted_summary"],
        }

    def forward(self, fc_feats, att_feats, seq=None, att_masks=None, retrieval_batch=None, opt=None, mode="forward"):
        self._active_retrieval_context = self._prepare_retrieval_context(retrieval_batch)
        try:
            if mode == "forward":
                return self.hatnet(fc_feats, att_feats, seq, att_masks)
            elif mode == "sample":
                return self.hatnet(fc_feats, att_feats, att_masks, opt=opt or {}, mode="sample")
            else:
                raise ValueError(f"Unsupported mode: {mode}")
        finally:
            self._active_retrieval_context = None

In [17]:
# 19. Forward, Backward, Prompt, and Checkpoint Reload Tests (Models A, B, C)
import captioning.modules.losses as losses

def build_base_opt():
    return SimpleNamespace(
        data_path=ExperimentConfig.OUT_DATA_DIR,
        input_json=INFO_JSON_PATH,
        input_label_h5=H5_LABEL_PATH,
        input_att_dir=ExperimentConfig.FEATURES_DIR,
        input_fc_dir="",
        input_box_dir="",
        batch_size=ExperimentConfig.BATCH_SIZE,
        seq_per_img=ExperimentConfig.SEQ_PER_IMG,
        max_length=ExperimentConfig.MAX_CAPTION_LEN,
        caption_model=ExperimentConfig.CAPTION_MODEL,
        d_model=ExperimentConfig.D_MODEL,
        d_ff=ExperimentConfig.D_FF,
        num_att_heads=ExperimentConfig.NUM_HEADS,
        num_layers=ExperimentConfig.N_ENC,
        N_enc=ExperimentConfig.N_ENC,
        N_dec=ExperimentConfig.N_DEC,
        dropout=ExperimentConfig.DROPOUT,
        input_encoding_size=ExperimentConfig.D_MODEL,
        rnn_size=ExperimentConfig.D_FF,
        drop_prob_lm=ExperimentConfig.DROP_PROB_LM,
        use_bn=0,
        seq_length=ExperimentConfig.MAX_CAPTION_LEN,
        eos_idx=0,
        pad_idx=0,
        fc_feat_size=0,
        att_feat_size=ExperimentConfig.ATT_FEAT_SIZE,
        att_hid_size=ExperimentConfig.D_MODEL,
        learning_rate=ExperimentConfig.LEARNING_RATE,
        use_box=0,
        use_fc=False,
        use_att=True,
        norm_att_feat=0,
        loader_num_workers=0,
        vocab_size=VOCAB_SIZE,
        vocab=ITOW,
    )

def run_model_prompt_and_checkpoint_tests():
    opt_test = build_base_opt()
    crit = losses.LanguageModelCriterion()
    
    # Prepare a real 2-image mini-batch from cached training features
    test_img_ids = SPLIT_MANIFEST["train_ids"][:2]
    img_feats = []
    for iid in test_img_ids:
        with np.load(os.path.join(ExperimentConfig.FEATURES_DIR, f"{iid}.npz")) as d:
            img_feats.append(d["feat"])
    img_att = torch.from_numpy(np.stack(img_feats, axis=0)).to(DEVICE)  # (2, 256, 3840)
    # Expand by seq_per_img=5 as DataLoader does in training
    att_expanded = img_att.unsqueeze(1).repeat(1, 5, 1, 1).view(10, 256, 3840)
    fc_expanded = torch.zeros(10, 0, device=DEVICE)
    fake_labels = torch.randint(1, VOCAB_SIZE, (10, ExperimentConfig.MAX_CAPTION_LEN + 2), device=DEVICE)
    fake_labels[:, 0] = 0
    fake_masks = torch.ones_like(fake_labels, dtype=torch.float32)
    
    ret_batch = RETRIEVAL_MEMORY.retrieve_batch(img_att, query_image_ids=test_img_ids, is_training=True, top_k=4)
    
    # --- Test Model A (Baseline) ---
    model_a = RetrievalEnhancedHATNet(opt_test, use_retrieval=False, ab_mode="baseline").to(DEVICE)
    out_a = model_a(fc_expanded, att_expanded, fake_labels[:, :-1], None, retrieval_batch=None)
    loss_a = crit(out_a, fake_labels[:, 1:], fake_masks[:, 1:])
    assert torch.isfinite(loss_a)
    loss_a.backward()
    assert model_a.hatnet.model.decoder2.layers[0].yita.grad is not None
    
    # Ensure baseline rejects accidental retrieval_batch
    try:
        model_a(fc_expanded, att_expanded, fake_labels[:, :-1], None, retrieval_batch=ret_batch)
        raise AssertionError("Model A failed to reject accidental retrieval_batch!")
    except RuntimeError:
        pass
        
    # --- Test Model B (Retrieval Only) ---
    model_b = RetrievalEnhancedHATNet(opt_test, use_retrieval=True, ab_mode="retrieval_only").to(DEVICE)
    out_b = model_b(fc_expanded, att_expanded, fake_labels[:, :-1], None, retrieval_batch=ret_batch)
    loss_b = crit(out_b, fake_labels[:, 1:], fake_masks[:, 1:])
    loss_b.backward()
    assert model_b.caption_encoder.token_embed.weight.grad is not None
    assert model_b.prompt_fusion.fusion_alpha.grad is not None
    
    # --- Test Model C (Full ViPCap Prompt Mapper + Fusion) ---
    model_c = RetrievalEnhancedHATNet(opt_test, use_retrieval=True, ab_mode="full_vipcap").to(DEVICE)
    optim_c = torch.optim.Adam(model_c.parameters(), lr=1e-3)
    optim_c.zero_grad()
    out_c = model_c(fc_expanded, att_expanded, fake_labels[:, :-1], None, retrieval_batch=ret_batch)
    loss_c = crit(out_c, fake_labels[:, 1:], fake_masks[:, 1:])
    assert torch.isfinite(loss_c)
    loss_c.backward()
    
    assert model_c.prompt_mapper.prompt_queries.grad is not None and torch.isfinite(model_c.prompt_mapper.prompt_queries.grad).all()
    assert model_c.prompt_fusion.fusion_alpha.grad is not None and torch.isfinite(model_c.prompt_fusion.fusion_alpha.grad)
    assert model_c.hatnet.model.decoder2.layers[0].yita.grad is not None
    
    q_before = model_c.prompt_mapper.prompt_queries.detach().clone()
    optim_c.step()
    assert torch.abs(model_c.prompt_mapper.prompt_queries - q_before).sum().item() > 0
    
    # --- Safe PyTorch 2.6+ Checkpoint Save & Reload Determinism Test ---
    model_c.eval()
    with torch.no_grad():
        ref_out = model_c(fc_expanded[:2], img_att, fake_labels[:2, :-1], None, retrieval_batch=ret_batch)
    tmp_ckpt = os.path.join(ExperimentConfig.CHECKPOINT_DIR, "_unit_test_ckpt.pth")
    torch.save({"model_state_dict": model_c.state_dict()}, tmp_ckpt)
    
    loaded_state = torch.load(tmp_ckpt, map_location=DEVICE, weights_only=True)
    fresh_c = RetrievalEnhancedHATNet(opt_test, use_retrieval=True, ab_mode="full_vipcap").to(DEVICE)
    fresh_c.load_state_dict(loaded_state["model_state_dict"])
    fresh_c.eval()
    with torch.no_grad():
        reload_out = fresh_c(fc_expanded[:2], img_att, fake_labels[:2, :-1], None, retrieval_batch=ret_batch)
    max_diff = torch.max(torch.abs(ref_out - reload_out)).item()
    assert max_diff < 1e-6, f"Reloaded model output diverged: {max_diff}"
    os.remove(tmp_ckpt)
    print(f"MODEL A/B/C FORWARD, BACKWARD, GRADIENT & CHECKPOINT RELOAD TESTS PASSED (max_diff={max_diff:.2e}).")

run_model_prompt_and_checkpoint_tests()

C:\Users\LENOVO\anaconda3\Lib\site-packages\torch\nn\modules\linear.py:124: UserWarning: Initializing zero-element tensors is a no-op
  init.kaiming_uniform_(self.weight, a=math.sqrt(5))


MODEL A/B/C FORWARD, BACKWARD, GRADIENT & CHECKPOINT RELOAD TESTS PASSED (max_diff=0.00e+00).


In [18]:
# 20. Consistent Evaluation Pipeline & Evaluator Unit Tests
import copy
import numpy as np
import torch
from pycocotools.coco import COCO
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider
from nltk.translate.meteor_score import meteor_score
import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

class ConsistentCaptionEvaluator:
    """Component L: One fixed, deterministic evaluation pipeline used identically for
    validation and final testing. Never silently swaps tokenizers or fabricates scores."""
    VERSION = "1.0.0-fixed-karpathy-coco"

    @staticmethod
    def decode_tokens(vocab: dict, seq: torch.Tensor) -> list:
        N, D = seq.size()
        captions = []
        for i in range(N):
            words = []
            for j in range(D):
                ix = int(seq[i, j].item())
                if ix > 0:
                    w = vocab.get(str(ix), vocab.get(ix, ""))
                    if w and w != "UNK":
                        words.append(w)
                else:
                    break
            captions.append(" ".join(words) if words else "aerial view")
        return captions

    @classmethod
    def evaluate_predictions(cls, coco_gt_file: str, predictions: list, expected_ids: set, try_spice: bool = False) -> dict:
        assert isinstance(predictions, list) and len(predictions) > 0, "Predictions list is empty!"
        pred_ids = [p["image_id"] for p in predictions]
        if len(pred_ids) != len(set(pred_ids)):
            raise ValueError("Duplicate image_id detected in predictions!")
        pred_ids_set = set(int(x) for x in pred_ids)
        expected_ids_set = set(int(x) for x in expected_ids)
        if pred_ids_set != expected_ids_set:
            raise ValueError(f"Prediction IDs mismatch ground truth! Diff={len(pred_ids_set ^ expected_ids_set)}")
            
        for p in predictions:
            cap = p.get("caption", None)
            if not isinstance(cap, str) or len(cap.strip()) == 0:
                raise ValueError(f"Empty or non-string caption for image_id {p.get('image_id')}!")
                
        coco_gt = COCO(coco_gt_file)
        gts = {}
        res = {}
        pred_map = {int(p["image_id"]): " ".join(normalize_caption_tokens(p["caption"])) for p in predictions}
        
        for iid in sorted(expected_ids_set):
            gts[iid] = [" ".join(normalize_caption_tokens(a["caption"])) for a in coco_gt.imgToAnns[iid]]
            res[iid] = [pred_map[iid]]
            if not res[iid][0]:
                raise ValueError(f"Normalized caption became empty for image_id {iid}!")
                
        raw_metrics = {}
        # 1. BLEU-1..4
        bleu_scores, _ = Bleu(4).compute_score(gts, res)
        for n, b_val in zip(["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"], bleu_scores):
            raw_metrics[n] = float(b_val)
            
        # 2. METEOR (Deterministic NLTK WordNet implementation; never silently skipped)
        meteor_vals = [
            meteor_score([ref.split() for ref in gts[iid]], res[iid][0].split())
            for iid in sorted(expected_ids_set)
        ]
        raw_metrics["METEOR"] = float(np.mean(meteor_vals))
        
        # 3. ROUGE-L
        rouge_val, _ = Rouge().compute_score(gts, res)
        raw_metrics["ROUGE_L"] = float(rouge_val)
        
        # 4. CIDEr
        cider_val, _ = Cider().compute_score(gts, res)
        raw_metrics["CIDEr"] = float(cider_val)
        
        for k, v in raw_metrics.items():
            assert np.isfinite(v), f"Metric {k} is non-finite: {v}"
            
        # 5. Optional SPICE (handled independently so Java absence never alters CIDEr/BLEU/METEOR/ROUGE_L)
        if try_spice:
            try:
                from pycocoevalcap.spice.spice import Spice
                spice_val, _ = Spice().compute_score(gts, res)
                raw_metrics["SPICE"] = float(spice_val)
            except Exception as e:
                raw_metrics["SPICE"] = None
                raw_metrics["SPICE_error"] = f"{type(e).__name__}: {str(e)}"
        return raw_metrics

def run_evaluator_unit_tests():
    val_ids_list = SPLIT_MANIFEST["val_ids"]
    valid_preds = [{"image_id": int(iid), "caption": "many planes are parked at the airport"} for iid in val_ids_list]
    m1 = ConsistentCaptionEvaluator.evaluate_predictions(VAL_COCO_JSON, valid_preds, set(val_ids_list))
    m2 = ConsistentCaptionEvaluator.evaluate_predictions(VAL_COCO_JSON, valid_preds, set(val_ids_list))
    assert m1 == m2 and "CIDEr" in m1 and "METEOR" in m1
    
    # Reject duplicate predictions
    try:
        ConsistentCaptionEvaluator.evaluate_predictions(VAL_COCO_JSON, valid_preds + [valid_preds[0]], set(val_ids_list))
        raise AssertionError("Failed to reject duplicate predictions!")
    except ValueError:
        pass
        
    # Reject empty caption
    bad_preds = copy.deepcopy(valid_preds)
    bad_preds[0]["caption"] = "   "
    try:
        ConsistentCaptionEvaluator.evaluate_predictions(VAL_COCO_JSON, bad_preds, set(val_ids_list))
        raise AssertionError("Failed to reject empty caption!")
    except ValueError:
        pass
    print("EVALUATION CONSISTENCY UNIT TESTS PASSED.")

run_evaluator_unit_tests()


loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
{'testlen': 2100, 'reflen': 2859, 'guess': [2100, 1800, 1500, 1200], 'correct': [627, 49, 15, 5]}
ratio: 0.7345225603355249
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
{'testlen': 2100, 'reflen': 2859, 'guess': [2100, 1800, 1500, 1200], 'correct': [627, 49, 15, 5]}
ratio: 0.7345225603355249
EVALUATION CONSISTENCY UNIT TESTS PASSED.


In [19]:
# 21. Complete Training, Validation, Checkpointing & Inference Engine
import sys
for mod in list(sys.modules.keys()):
    if mod.startswith("captioning"):
        del sys.modules[mod]

from captioning.data.dataloader import DataLoader

@torch.no_grad()
def generate_split_predictions(
    model: RetrievalEnhancedHATNet,
    split_image_ids: list,
    retrieval_memory: TrainingOnlyRetrievalMemory,
    beam_size: int = 3,
    eval_batch_size: int = 32,
) -> list:
    model.eval()
    predictions = []
    for i in range(0, len(split_image_ids), eval_batch_size):
        batch_ids = [int(x) for x in split_image_ids[i : i + eval_batch_size]]
        feats = []
        for iid in batch_ids:
            with np.load(os.path.join(ExperimentConfig.FEATURES_DIR, f"{iid}.npz")) as d:
                feats.append(d["feat"])
        att_batch = torch.from_numpy(np.stack(feats, axis=0)).to(DEVICE)
        fc_batch = torch.zeros(att_batch.shape[0], 0, device=DEVICE)
        
        ret_batch = None
        if model.use_retrieval:
            ret_batch = retrieval_memory.retrieve_batch(
                att_batch, query_image_ids=batch_ids, is_training=False, top_k=ExperimentConfig.RETRIEVAL_TOP_K
            )
        seq, _ = model(
            fc_batch, att_batch, retrieval_batch=ret_batch,
            opt={"beam_size": beam_size, "sample_n": 1}, mode="sample"
        )
        decoded = ConsistentCaptionEvaluator.decode_tokens(model.opt.vocab, seq)
        for iid, cap in zip(batch_ids, decoded):
            predictions.append({"image_id": int(iid), "caption": cap.strip()})
    return predictions


def train_and_evaluate_model(
    run_name: str,
    use_retrieval: bool,
    ab_mode: str,
    max_epochs: int,
    max_steps_per_epoch: int = None,
) -> dict:
    set_all_seeds(ExperimentConfig.SEED)
    opt = build_base_opt()
    loader = DataLoader(opt)
    opt.vocab_size = loader.dataset.get_vocab_size()
    opt.seq_length = loader.dataset.get_seq_length()
    opt.vocab = {str(k): v for k, v in loader.dataset.get_vocab().items()}
    
    model = RetrievalEnhancedHATNet(
        opt,
        use_retrieval=use_retrieval,
        ab_mode=ab_mode,
        num_prompt_tokens=ExperimentConfig.NUM_PROMPT_TOKENS,
    ).to(DEVICE)
    
    criterion = losses.LanguageModelCriterion()
    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad],
        lr=ExperimentConfig.LEARNING_RATE,
        betas=(0.9, 0.98),
        eps=1e-9
    )
    
    train_ids_list = SPLIT_MANIFEST["train_ids"]
    val_ids_list = SPLIT_MANIFEST["val_ids"]
    steps_per_epoch = int(math.ceil(len(train_ids_list) / ExperimentConfig.BATCH_SIZE))
    total_steps = steps_per_epoch * max_epochs
    warmup_steps = min(ExperimentConfig.WARMUP_STEPS, max(1, total_steps // 5))
    
    def lr_lambda(step: int) -> float:
        if step < warmup_steps:
            return float(step + 1) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return max(0.05, 0.5 * (1.0 + math.cos(math.pi * progress)))
        
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    
    best_val_cider = -1.0
    best_epoch = -1
    history = []
    
    best_ckpt_path = os.path.join(ExperimentConfig.CHECKPOINT_DIR, f"{run_name}_best.pth")
    latest_ckpt_path = os.path.join(ExperimentConfig.CHECKPOINT_DIR, f"{run_name}_latest.pth")
    
    retrieval_meta_payload = getattr(RETRIEVAL_MEMORY, "metadata", None) if use_retrieval else None
    iters_per_epoch = steps_per_epoch if max_steps_per_epoch is None else max_steps_per_epoch
    
    print(f"\n--- Starting Run '{run_name}' (use_retrieval={use_retrieval}, ab_mode='{ab_mode}', epochs={max_epochs}) ---")
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        model.train()
        loader.reset_iterator("train")
        train_losses = []
        
        for step in range(1, iters_per_epoch + 1):
            optimizer.zero_grad()
            batch = loader.get_batch("train")
            att_feats = torch.as_tensor(batch["att_feats"], device=DEVICE)
            labels = torch.as_tensor(batch["labels"], dtype=torch.long, device=DEVICE)
            masks = torch.as_tensor(batch["masks"], device=DEVICE)
            att_masks = torch.as_tensor(batch["att_masks"], device=DEVICE) if batch.get("att_masks") is not None else None
            fc_feats = torch.zeros(att_feats.shape[0], 0, device=DEVICE)
            
            ret_batch = None
            if model.use_retrieval:
                batch_img_ids = [int(info["id"]) for info in batch["infos"]]
                unique_img_att = att_feats if att_feats.shape[0] == len(batch_img_ids) else att_feats[0::opt.seq_per_img]
                ret_batch = RETRIEVAL_MEMORY.retrieve_batch(
                    unique_img_att,
                    query_image_ids=batch_img_ids,
                    is_training=True,
                    top_k=ExperimentConfig.RETRIEVAL_TOP_K,
                )
                
            # FIXED: Pass full 3D `labels` into model, and slice last axis (`...`) for targets
            out = model(fc_feats, att_feats, labels, att_masks, retrieval_batch=ret_batch, mode="forward")
            loss = criterion(out, labels[..., 1:], masks[..., 1:])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), ExperimentConfig.GRAD_CLIP_NORM)
            optimizer.step()
            scheduler.step()
            train_losses.append(float(loss.item()))
            
        mean_train_loss = float(np.mean(train_losses))
        
        # --- End-of-Epoch Validation Loss ---
        model.eval()
        loader.reset_iterator("val")
        val_loss_sum = 0.0
        val_evaluated = 0
        val_limit = len(val_ids_list) if max_steps_per_epoch is None else min(len(val_ids_list), 32)
        
        with torch.no_grad():
            while val_evaluated < val_limit:
                v_batch = loader.get_batch("val")
                rem = val_limit - val_evaluated
                cur_imgs = min(ExperimentConfig.BATCH_SIZE, rem, len(v_batch["infos"]))
                
                v_att = torch.as_tensor(v_batch["att_feats"][:cur_imgs], device=DEVICE)
                v_lbl = torch.as_tensor(v_batch["labels"][:cur_imgs], dtype=torch.long, device=DEVICE)
                v_msk = torch.as_tensor(v_batch["masks"][:cur_imgs], device=DEVICE)
                v_fc = torch.zeros(v_att.shape[0], 0, device=DEVICE)
                
                v_ret = None
                if model.use_retrieval:
                    v_img_ids = [int(info["id"]) for info in v_batch["infos"][:cur_imgs]]
                    v_unique_att = v_att if v_att.shape[0] == len(v_img_ids) else v_att[0::opt.seq_per_img]
                    v_ret = RETRIEVAL_MEMORY.retrieve_batch(
                        v_unique_att, query_image_ids=v_img_ids, is_training=False
                    )
                # FIXED: Pass full 3D `v_lbl` into model, and slice last axis (`...`) for targets
                v_out = model(v_fc, v_att, v_lbl, None, retrieval_batch=v_ret, mode="forward")
                v_loss = criterion(v_out, v_lbl[..., 1:], v_msk[..., 1:])
                val_loss_sum += float(v_loss.item()) * cur_imgs
                val_evaluated += cur_imgs
                
        mean_val_loss = float(val_loss_sum / max(1, val_evaluated))
        
        # --- Validation Caption Generation & Consistent Metric Calculation ---
        val_preds = generate_split_predictions(
            model, val_ids_list, RETRIEVAL_MEMORY, beam_size=ExperimentConfig.BEAM_SIZE
        )
        val_metrics = ConsistentCaptionEvaluator.evaluate_predictions(
            VAL_COCO_JSON, val_preds, set(val_ids_list), try_spice=False
        )
        raw_val_cider = float(val_metrics["CIDEr"])
        elapsed = time.time() - t0
        
        is_best = raw_val_cider > best_val_cider
        if is_best:
            best_val_cider = raw_val_cider
            best_epoch = epoch
            
        epoch_record = {
            "epoch": epoch,
            "train_loss": mean_train_loss,
            "val_loss": mean_val_loss,
            "val_metrics_raw": val_metrics,
            "best_val_cider_raw": best_val_cider,
            "best_epoch": best_epoch,
            "epoch_time_sec": elapsed,
        }
        history.append(epoch_record)
        
        ckpt_payload = {
            "run_name": run_name,
            "epoch": int(epoch),
            "best_epoch": int(best_epoch),
            "best_val_cider_raw": float(best_val_cider),
            "use_retrieval": bool(use_retrieval),
            "ab_mode": str(ab_mode),
            "num_prompt_tokens": int(ExperimentConfig.NUM_PROMPT_TOKENS),
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "opt_config": {k: v for k, v in vars(opt).items() if isinstance(v, (int, float, str, bool, list, dict)) or v is None},
            "retrieval_metadata": retrieval_meta_payload,
        }
        tmp_latest = latest_ckpt_path + ".tmp"
        torch.save(ckpt_payload, tmp_latest)
        os.replace(tmp_latest, latest_ckpt_path)
        
        if is_best:
            tmp_best = best_ckpt_path + ".tmp"
            torch.save(ckpt_payload, tmp_best)
            os.replace(tmp_best, best_ckpt_path)
            with open(ExperimentConfig.VAL_PREDS_JSON, "w", encoding="utf-8") as f:
                json.dump(val_preds, f, indent=2)
            if run_name == "model_c_full_vipcap":
                shutil.copyfile(best_ckpt_path, os.path.join(ExperimentConfig.CHECKPOINT_DIR, "best_cider_model.pth"))
                shutil.copyfile(latest_ckpt_path, os.path.join(ExperimentConfig.CHECKPOINT_DIR, "latest_checkpoint.pth"))
                
        tag = " *BEST*" if is_best else ""
        print(
            f"[{run_name}] Epoch {epoch:02d}/{max_epochs} ({elapsed:.1f}s) | "
            f"Train Loss: {mean_train_loss:.4f} | Val Loss: {mean_val_loss:.4f} | "
            f"Val BLEU-4: {val_metrics['Bleu_4']*100:.2f} | Val CIDEr (raw): {raw_val_cider:.4f} ({raw_val_cider*100:.2f}){tag}"
        )
        if epoch == 1:
            print(f"   Sample Val Caption [0]: '{val_preds[0]['caption']}'")
        
    return {
        "run_name": run_name,
        "use_retrieval": use_retrieval,
        "ab_mode": ab_mode,
        "best_epoch": best_epoch,
        "best_val_cider_raw": best_val_cider,
        "best_ckpt_path": best_ckpt_path,
        "latest_ckpt_path": latest_ckpt_path,
        "history": history,
    }

In [20]:
# 22. Integration Smoke Tests (Tiny Baseline & Tiny Retrieval-Enhanced Runs)
print("Running mandatory integration smoke test (1 epoch, 2 steps) for Baseline and Retrieval-Enhanced models...")
smoke_base_res = train_and_evaluate_model(
    run_name="smoke_baseline", use_retrieval=False, ab_mode="baseline", max_epochs=1, max_steps_per_epoch=2
)
smoke_enh_res = train_and_evaluate_model(
    run_name="smoke_enhanced", use_retrieval=True, ab_mode="full_vipcap", max_epochs=1, max_steps_per_epoch=2
)
assert os.path.exists(smoke_base_res["best_ckpt_path"]) and os.path.exists(smoke_enh_res["best_ckpt_path"])
print("INTEGRATION SMOKE TESTS PASSED FOR BOTH BASELINE AND RETRIEVAL-ENHANCED MODELS.")

Running mandatory integration smoke test (1 epoch, 2 steps) for Baseline and Retrieval-Enhanced models...
DataLoader loading json file:  C:\Users\LENOVO\Desktop\vansh\data\rsicd_info.json
vocab size is  1241
DataLoader loading h5 file:   C:\Users\LENOVO\Desktop\vansh\features  C:\Users\LENOVO\Desktop\vansh\data\rsicd_label.h5
max sequence length in data is 20
read 3000 image features
assigned 2400 images to split train
assigned 300 images to split val
assigned 300 images to split test

--- Starting Run 'smoke_baseline' (use_retrieval=False, ab_mode='baseline', epochs=1) ---
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
{'testlen': 600, 'reflen': 2834, 'guess': [600, 300, 0, 0], 'correct': [1, 0, 0, 0]}
ratio: 0.21171489061389848
[smoke_baseline] Epoch 01/1 (48.9s) | Train Loss: 7.4513 | Val Loss: 6.8955 | Val BLEU-4: 0.00 | Val CIDEr (raw): 0.0004 (0.04) *BEST*
   Sample Val Caption [0]: 'aerial view'
DataLoader loading json file:  C:\Users\LENOVO\D

In [22]:
# 23. Full Training & Ablation Execution (Model A: Baseline, Model B: Retrieval-Only, Model C: Full ViPCap)
ALL_EXPERIMENTS = {}

if ExperimentConfig.RUN_FULL_TRAINING:
    # Model A: Original HATNet Baseline
    ALL_EXPERIMENTS["Model_A_Baseline"] = train_and_evaluate_model(
        run_name="model_a_baseline",
        use_retrieval=False,
        ab_mode="baseline",
        max_epochs=ExperimentConfig.MAX_EPOCHS,
    )
    
    # Model B: HATNet + Retrieval Conditioning (without ViPCap Prompt Mapper)
    ALL_EXPERIMENTS["Model_B_Retrieval_Only"] = train_and_evaluate_model(
        run_name="model_b_retrieval_only",
        use_retrieval=True,
        ab_mode="retrieval_only",
        max_epochs=ExperimentConfig.MAX_EPOCHS,
    )
    
    # Model C: HATNet + Retrieval Conditioning + ViPCap Visual Prompt Fusion
    ALL_EXPERIMENTS["Model_C_Full_ViPCap"] = train_and_evaluate_model(
        run_name="model_c_full_vipcap",
        use_retrieval=True,
        ab_mode="full_vipcap",
        max_epochs=ExperimentConfig.MAX_EPOCHS,
    )
else:
    ALL_EXPERIMENTS["Model_A_Baseline"] = smoke_base_res
    ALL_EXPERIMENTS["Model_C_Full_ViPCap"] = smoke_enh_res

with open(ExperimentConfig.TRAINING_HISTORY_JSON, "w", encoding="utf-8") as f:
    json.dump(ALL_EXPERIMENTS, f, indent=2)
print(f"Saved complete training history to {ExperimentConfig.TRAINING_HISTORY_JSON}")

DataLoader loading json file:  C:\Users\LENOVO\Desktop\vansh\data\rsicd_info.json
vocab size is  1241
DataLoader loading h5 file:   C:\Users\LENOVO\Desktop\vansh\features  C:\Users\LENOVO\Desktop\vansh\data\rsicd_label.h5
max sequence length in data is 20
read 3000 image features
assigned 2400 images to split train
assigned 300 images to split val
assigned 300 images to split test

--- Starting Run 'model_a_baseline' (use_retrieval=False, ab_mode='baseline', epochs=30) ---
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
{'testlen': 2700, 'reflen': 2989, 'guess': [2700, 2400, 2100, 1800], 'correct': [1375, 481, 221, 105]}
ratio: 0.9033121445296409
[model_a_baseline] Epoch 01/30 (304.9s) | Train Loss: 4.6769 | Val Loss: 2.9806 | Val BLEU-4: 14.22 | Val CIDEr (raw): 0.2920 (29.20) *BEST*
   Sample Val Caption [0]: 'many buildings and green trees are around a viaduct'
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
{'tes

In [23]:
# 24. Final Test Evaluation & Fair Model A vs. Model B vs. Model C Comparison
def evaluate_saved_checkpoint_on_test(ckpt_path: str, memory: TrainingOnlyRetrievalMemory) -> tuple:
    # Verify memory integrity prior to test evaluation
    assert memory.train_ids_set.isdisjoint(VAL_IDS_SET)
    assert memory.train_ids_set.isdisjoint(TEST_IDS_SET)
    
    try:
        ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
    except Exception:
        ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    opt_loaded = SimpleNamespace(**ckpt["opt_config"])
    fresh_model = RetrievalEnhancedHATNet(
        opt_loaded,
        use_retrieval=ckpt["use_retrieval"],
        ab_mode=ckpt["ab_mode"],
        num_prompt_tokens=ckpt["num_prompt_tokens"],
    ).to(DEVICE)
    fresh_model.load_state_dict(ckpt["model_state_dict"])
    fresh_model.eval()
    
    test_ids_list = SPLIT_MANIFEST["test_ids"]
    assert len(test_ids_list) == ExperimentConfig.TEST_COUNT
    
    test_preds = generate_split_predictions(
        fresh_model, test_ids_list, memory, beam_size=ExperimentConfig.BEAM_SIZE
    )
    assert len(test_preds) == ExperimentConfig.TEST_COUNT
    assert set(p["image_id"] for p in test_preds) == set(test_ids_list)
    
    test_metrics = ConsistentCaptionEvaluator.evaluate_predictions(
        TEST_COCO_JSON, test_preds, set(test_ids_list), try_spice=True
    )
    return test_metrics, test_preds, ckpt

TEST_COMPARISON_RESULTS = {}
for label, exp_info in ALL_EXPERIMENTS.items():
    t_metrics, t_preds, ckpt_meta = evaluate_saved_checkpoint_on_test(exp_info["best_ckpt_path"], RETRIEVAL_MEMORY)
    TEST_COMPARISON_RESULTS[label] = {
        "best_epoch": ckpt_meta["best_epoch"],
        "best_val_cider_raw": ckpt_meta["best_val_cider_raw"],
        "test_metrics_raw": t_metrics,
    }
    if "Full_ViPCap" in label:
        with open(ExperimentConfig.TEST_PREDS_JSON, "w", encoding="utf-8") as f:
            json.dump(t_preds, f, indent=2)

with open(ExperimentConfig.TEST_METRICS_JSON, "w", encoding="utf-8") as f:
    json.dump(TEST_COMPARISON_RESULTS, f, indent=2)

# Print formatted scientific comparison table
metric_keys = ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4", "METEOR", "ROUGE_L", "CIDEr"]
header = f"{'Model Configuration':26s} | {'BestEp':>6s} | " + " | ".join(f"{m:>7s}" for m in metric_keys)
print("\n" + "=" * len(header))
print(header)
print("-" * len(header))
base_scores = TEST_COMPARISON_RESULTS.get("Model_A_Baseline", {}).get("test_metrics_raw", {})
for label, res in TEST_COMPARISON_RESULTS.items():
    tm = res["test_metrics_raw"]
    row_str = f"{label:26s} | {res['best_epoch']:6d} | " + " | ".join(f"{tm[m]*100:7.2f}" for m in metric_keys)
    print(row_str)
    if label != "Model_A_Baseline" and base_scores:
        diff_str = f"{'  (Delta vs. Model A)':26s} | {'':6s} | " + " | ".join(
            f"{(tm[m] - base_scores[m])*100:+7.2f}" for m in metric_keys
        )
        print(diff_str)
print("=" * len(header))

summary_lines = [
    "Retrieval-Enhanced HATNet Run Summary",
    f"Timestamp: {datetime.datetime.now().isoformat()}",
    f"Dataset Fingerprint: {DATASET_FINGERPRINT}",
    f"Split Fingerprint: {SPLIT_MANIFEST['split_fingerprint']}",
    f"Evaluator Version: {ConsistentCaptionEvaluator.VERSION}",
    json.dumps(TEST_COMPARISON_RESULTS, indent=2),
]
with open(ExperimentConfig.RUN_SUMMARY_TXT, "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))
print(f"Saved final test metrics to {ExperimentConfig.TEST_METRICS_JSON} and summary to {ExperimentConfig.RUN_SUMMARY_TXT}.")


loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
{'testlen': 2910, 'reflen': 3043, 'guess': [2910, 2610, 2310, 2010], 'correct': [2074, 1243, 788, 510]}
ratio: 0.9562931317775365
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
{'testlen': 2948, 'reflen': 3057, 'guess': [2948, 2648, 2348, 2048], 'correct': [2146, 1324, 865, 584]}
ratio: 0.9643441282299757
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
{'testlen': 2857, 'reflen': 3015, 'guess': [2857, 2557, 2257, 1957], 'correct': [2108, 1316, 872, 583]}
ratio: 0.9475953565502662

Model Configuration        | BestEp |  Bleu_1 |  Bleu_2 |  Bleu_3 |  Bleu_4 |  METEOR | ROUGE_L |   CIDEr
---------------------------------------------------------------------------------------------------------
Model_A_Baseline           |     10 |   68.09 |   55.66 |   46.56 |   39.55 |   58.61 |   58.33 |  215.82
Model_B_Retrieval_Only     |      8 |   70